#### NSR Project Data Wrangling

In [1]:
# import necessary packages
import pandas as pd
import numpy as np
import os

In [2]:
pd.set_option('display.max_rows',None)
pd.set_option('display.max_rows',None)

In [3]:
# open csv files as pandas dataframes 

start_y = 2011
max_y = 2024
curr_y = start_y

path = '~/desktop/projects/nsr_project/static/NSRData_CSV/'
data = {}

while curr_y <= max_y: 
    file_name = 'nsr_' + str(curr_y) + '.csv'
    df = pd.read_csv(os.path.join(path,file_name))
    data[curr_y] = df
    curr_y +=1

#### Define functions for use later

In [4]:
# Do dataset wide cleanup first?
# def functions to do some basic cleanup
def basic_clean(df):

    # replace character that make data less legible 
    df = df.replace('\n',' ', regex=True)
    df = df.replace(['«','»'], '"',regex=True)
    df = df.replace(',','.', regex=True)
    df = df.replace('-',np.nan)

    # signal non-ASCII characters within the data (to find Russian language characters that need to be translated
    n=0
    for row_index, row in df.iterrows():
        for col_name, cell_val in row.items():
            if df[col_name].dtype == object:
                if not str(cell_val).isascii():
                    n+=1
                    print('[',col_name,row_index,']')        # give location of non-ASCII characters
    print('New line characters have been removed. Detected', n, 'cell(s) with non-ASCII characters.\nNull Count:\n',df.isnull().sum())
    
    return df

In [5]:
# helper function that allows for more eloquent conversion 
def date_conversion_helper(to_change_dict):
    dt_list = []
    for key, item in to_change_dict:
        for s in item:
            s = pd.to_datetime(s,format=key)         # key is pattern to parse with
            dt_list.append(s)
    return dt_list


In [6]:
# helper function to more accurately split cells containing more than one cell worth of information
# takes in **dictionary of key, values pairs collating a regex pattern with a series
#    will the series need to be converted to ndarray?
# returns multiple series so that:
#    df[['series1','series2','series3']] = func({df[series0]:'regex pattern codes')

def splitter(series_dict):

    # dictionary that matches codes from argument to actually regex patterns
    pattern_dict = {'p':'','d':'','t':'',
                    'c':'','cm':''
                   }
    
    for series, codes in series_dict.items():
        split_codes = codes.split()

        for code in split_codes:
           # try: 
            series = series.str.extract(code)

    return 1



            

##### Set up column standardization

In [7]:
# 2011

df_2011 = data[2011]
df_2011.head()

,Unnamed: 0,0,1,2,3,4,5,6
0,0,NaN,Vessel’s Name,Type,Flag,Cargo,Port of Loading,Port of Destination
1,1,1.0,Perseverance,Tanker vessel,Singapore,Gas condensate,"Vitino, Russia","Ningbo,\nChina"
2,2,2.0,STI Heritage,Tanker vessel,Marshall\nIslands,Gas condensate,"Vitino, Russia","Map Ta Phut, Thailand"
3,3,3.0,Marilee,Tanker vessel,Norway,Gas condensate,"Vitino, Russia","Huizhou,\nChina"
4,4,4.0,Vladimir Tikhonov,Tanker vessel,Liberia,Gas condensate,"Honningsvåg,\nNorway","Map Ta Phut, Thailand"


In [8]:
df_2011.tail()

,Unnamed: 0,0,1,2,3,4,5,6
37,37,37.0,Ventspils,Tanker vessel,Russia,ballast,East,West
38,38,38.0,Mikhail Kutuzov,Bulker vessel,Russia,ballast,East,West
39,39,39.0,Ivan Kireev,Hydrographical\nvessel,Russia,ballast,Archangelsk,Slavyanka
40,40,40.0,Vengeri,tug/supply ship,Russia,ballast,West,East
41,41,41.0,Mar Adriana,Tanker vessel,Spain,ballast,Rotterdam,Vladivostok


In [9]:
df_2011.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  42 non-null     int64  
 1   0           41 non-null     float64
 2   1           42 non-null     object 
 3   2           42 non-null     object 
 4   3           42 non-null     object 
 5   4           42 non-null     object 
 6   5           42 non-null     object 
 7   6           42 non-null     object 
dtypes: float64(1), int64(1), object(6)
memory usage: 2.8+ KB


In [10]:
# make row index 0 into column headers
df_2011.columns = df_2011.iloc[0]
df_2011 = df_2011[1:]
df_2011.reset_index(drop=True, inplace=True)
df_2011.rename(columns=str)

df_2011.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   0                    41 non-null     int64  
 1   nan                  41 non-null     float64
 2   Vessel’s Name        41 non-null     object 
 3   Type                 41 non-null     object 
 4   Flag                 41 non-null     object 
 5   Cargo                41 non-null     object 
 6   Port of Loading      41 non-null     object 
 7   Port of Destination  41 non-null     object 
dtypes: float64(1), int64(1), object(6)
memory usage: 2.7+ KB


In [11]:
# change column header names
df_2011.columns = ['index','index2','vessel name','ship type','flag','cargo type','port of departure','port of arrival']
df_2011.columns

Index(['index', 'index2', 'vessel name', 'ship type', 'flag', 'cargo type',
       'port of departure', 'port of arrival'],
      dtype='object')

In [12]:
df_2011 = df_2011.drop(['index','index2'],axis=1)

In [13]:
df_2011.head()

,vessel name,ship type,flag,cargo type,port of departure,port of arrival
0,Perseverance,Tanker vessel,Singapore,Gas condensate,"Vitino, Russia","Ningbo,\nChina"
1,STI Heritage,Tanker vessel,Marshall\nIslands,Gas condensate,"Vitino, Russia","Map Ta Phut, Thailand"
2,Marilee,Tanker vessel,Norway,Gas condensate,"Vitino, Russia","Huizhou,\nChina"
3,Vladimir Tikhonov,Tanker vessel,Liberia,Gas condensate,"Honningsvåg,\nNorway","Map Ta Phut, Thailand"
4,Stena Poseidon,Tanker vessel,Finland,Gas condensate,"Vitino, Russia","Incheon,\nSouth Korea"


In [14]:
df_2011 = basic_clean(df_2011)

[ port of departure 3 ]
[ vessel name 29 ]
New line characters have been removed. Detected 2 cell(s) with non-ASCII characters.
Null Count:
 vessel name          0
ship type            0
flag                 0
cargo type           0
port of departure    0
port of arrival      0
dtype: int64


In [15]:
df_2011.head()

,vessel name,ship type,flag,cargo type,port of departure,port of arrival
0,Perseverance,Tanker vessel,Singapore,Gas condensate,Vitino. Russia,Ningbo. China
1,STI Heritage,Tanker vessel,Marshall Islands,Gas condensate,Vitino. Russia,Map Ta Phut. Thailand
2,Marilee,Tanker vessel,Norway,Gas condensate,Vitino. Russia,Huizhou. China
3,Vladimir Tikhonov,Tanker vessel,Liberia,Gas condensate,Honningsvåg. Norway,Map Ta Phut. Thailand
4,Stena Poseidon,Tanker vessel,Finland,Gas condensate,Vitino. Russia,Incheon. South Korea


In [16]:
# 2012
df_2012 = data[2012]

df_2012.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11
0,0,№,Vessel and Flag,Ice Class,Shipowner /\nOperator,Cargo,Destination,Port and Date\nof Sail,Entry to\nNSR,Exit from\nNSR,Time on NSR,"Average speed,\nknots",NaN
1,1,1,mt\n«Varzuga»\nRussia,Arc5,JSC MSCo,12538 t\nDiesel fuel,Pevek-Anadyr,Murmansk\n20.06.12\n22:30,23.06.12\n06:00,16.07.12\n10:00,"To Pevek 15.4 days\nFrom Pevek to\nDezhnev\n4,...","5,8.",NaN
2,2,2,Mt\n«Indiga»\nRussia,Arc5,JSC MSCo,12979 t\nDiesel Fuel,Pevek-Anadyr,Murmansk\n21.06.12\n22:00,24.06.12\n05:00,16.07.12\n10:00,"To Pevek 14,3 days\nFrom Pevek to\nDezhnev\n4,...","5,9",NaN
3,3,3,Supply vessel\n«Vengeri»\nRussia,Ice 1A\nSuper\n(Arc 5),FEMCO LTD,"repositioning,\ndspl 3733 t","K holmsk,\nSakhalin",Murmansk\n05.07.12\n13:00,08.07.12\n04:00,23.07.12\n00:00,"14,8","7,3",NaN
4,4,4,mt\n«Nordic\nOdyssey»\nPANAMA,Ice 1A\n(Arc4),Nordic Bulk\nCarriers,67520 t\nЖРК\n«Еврохим»,Китай,Murmansk\n10.07.12\n02:00,11.07.12\n22:00,23.07.12\n05:00,"11,2","9,7",NaN


In [17]:
df_2012.tail(10)

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11
52,52,45,icebreaker\n«Nordica»\nFINLAND,Polar-10\nIcebreaker\n(DNV),Arctia Offshore\nLtd.,Ballast\nFull\nDisplacement\n12 800 t,Denmark,"Alaska,\nUSA\n12.11.12\n08:30",13.11.12\n02:00,NaN,NaN,NaN,NaN
53,53,46,icebreaker\n«Fennica»\nFINLAND,Polar-10\nIcebreaker\n(DNV),Arctia Offshore\nLtd.,Ballast\nFull\nDisplacement\n12 811 t,Denmark,"Alaska,\nUSA\n12.11.12\n07:45",13.11.12\n02:00,NaN,NaN,NaN,NaN
54,54,Type of Cargo,Number\nof\nVessels,"Cargo\nVolume,\ntons","Full\nDisplacement,\nTons",Number\nof Vessels\nEastbound,"Cargo\nVolume\nEastbound,\ntons",Number of\nVessels\nWestbound,Cargo\nVolume\nWestbound,Number\nof Vessels\nin ballast\nEastbound,Full\nDisplacement\nEastbound,Number of\nVessels in\nballast\nWestbound,Full\nDisplacement\nWestbound
55,55,Liquid,26,894 079,NaN,18,661 326,8,232 753,NaN,NaN,NaN,NaN
56,56,Bulk,6,359 201,NaN,4,262 263,2,96 938,NaN,NaN,NaN,NaN
57,57,Fish,1,8 265,NaN,NaN,NaN,1,8 265,NaN,NaN,NaN,NaN
58,58,General,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,59,In Ballast,6,NaN,472 075,NaN,NaN,NaN,NaN,NaN,NaN,6,472 075
60,60,Repositioning,7,NaN,78 351,NaN,NaN,NaN,NaN,3,14 365,4,63 986
61,61,TOTAL:,46,1 261 545,550 426,22,923 589,11,337 956,3,14 365,10,536 061


In [18]:
df_2012_totals = df_2012[54:62]
df_2012_totals.head(10)

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11
54,54,Type of Cargo,Number\nof\nVessels,"Cargo\nVolume,\ntons","Full\nDisplacement,\nTons",Number\nof Vessels\nEastbound,"Cargo\nVolume\nEastbound,\ntons",Number of\nVessels\nWestbound,Cargo\nVolume\nWestbound,Number\nof Vessels\nin ballast\nEastbound,Full\nDisplacement\nEastbound,Number of\nVessels in\nballast\nWestbound,Full\nDisplacement\nWestbound
55,55,Liquid,26,894 079,NaN,18,661 326,8,232 753,NaN,NaN,NaN,NaN
56,56,Bulk,6,359 201,NaN,4,262 263,2,96 938,NaN,NaN,NaN,NaN
57,57,Fish,1,8 265,NaN,NaN,NaN,1,8 265,NaN,NaN,NaN,NaN
58,58,General,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,59,In Ballast,6,NaN,472 075,NaN,NaN,NaN,NaN,NaN,NaN,6,472 075
60,60,Repositioning,7,NaN,78 351,NaN,NaN,NaN,NaN,3,14 365,4,63 986
61,61,TOTAL:,46,1 261 545,550 426,22,923 589,11,337 956,3,14 365,10,536 061


In [19]:
df_2012 = df_2012[0:54]
df_2012.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11
49,49,42,Mv\n«Nordic\nOdyssey»\nPANAMA,Ice 1A\n(Arc4),Nordic Bulk\nCarriers,71 786 t\ncoal,"HAMBURG,\nGermany","VANCOUVER,\nCanada\n26.10.12\n04:00",04.11.12\n02:15,14.11.12\n04:00,10.0,10.4,NaN
50,50,43,mv\n«Nordic\nOrion»\nPANAMA,Ice-1A\n(Arc4),Nordic Bulk\nCarriers,Ballast\nFull\nDisplacement\n87216 t,Мурманск,"Caofeidian,\nChina\n28.10.12\n04:00",08.11.12\n06:30,18.11.12\n18:30,10.5,"10,3",NaN
51,51,44,LNG\n«Ob River»\nMarshall Island,Ice 1A\n(Arc4),LANCE\nSHIPPING S.A,"66 342 т.,\n134 738,5 м3,\nLNG","Tobata,\nJapan","Hammerfest,\nNorway\n07.11.12\n18:55",09.11.12\n22:00\nKara Gate,18.11.12\n21:20\nDezhnev Cape,9.0,"12,5",NaN
52,52,45,icebreaker\n«Nordica»\nFINLAND,Polar-10\nIcebreaker\n(DNV),Arctia Offshore\nLtd.,Ballast\nFull\nDisplacement\n12 800 t,Denmark,"Alaska,\nUSA\n12.11.12\n08:30",13.11.12\n02:00,NaN,NaN,NaN,NaN
53,53,46,icebreaker\n«Fennica»\nFINLAND,Polar-10\nIcebreaker\n(DNV),Arctia Offshore\nLtd.,Ballast\nFull\nDisplacement\n12 811 t,Denmark,"Alaska,\nUSA\n12.11.12\n07:45",13.11.12\n02:00,NaN,NaN,NaN,NaN


In [20]:
df_2012.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11
0,0,№,Vessel and Flag,Ice Class,Shipowner /\nOperator,Cargo,Destination,Port and Date\nof Sail,Entry to\nNSR,Exit from\nNSR,Time on NSR,"Average speed,\nknots",NaN
1,1,1,mt\n«Varzuga»\nRussia,Arc5,JSC MSCo,12538 t\nDiesel fuel,Pevek-Anadyr,Murmansk\n20.06.12\n22:30,23.06.12\n06:00,16.07.12\n10:00,"To Pevek 15.4 days\nFrom Pevek to\nDezhnev\n4,...","5,8.",NaN
2,2,2,Mt\n«Indiga»\nRussia,Arc5,JSC MSCo,12979 t\nDiesel Fuel,Pevek-Anadyr,Murmansk\n21.06.12\n22:00,24.06.12\n05:00,16.07.12\n10:00,"To Pevek 14,3 days\nFrom Pevek to\nDezhnev\n4,...","5,9",NaN
3,3,3,Supply vessel\n«Vengeri»\nRussia,Ice 1A\nSuper\n(Arc 5),FEMCO LTD,"repositioning,\ndspl 3733 t","K holmsk,\nSakhalin",Murmansk\n05.07.12\n13:00,08.07.12\n04:00,23.07.12\n00:00,"14,8","7,3",NaN
4,4,4,mt\n«Nordic\nOdyssey»\nPANAMA,Ice 1A\n(Arc4),Nordic Bulk\nCarriers,67520 t\nЖРК\n«Еврохим»,Китай,Murmansk\n10.07.12\n02:00,11.07.12\n22:00,23.07.12\n05:00,"11,2","9,7",NaN


In [21]:
# change column header names
# need to mold data into df.columns = ['index','index2','vessel name','type','flag','cargo','port of loading','port of destination']
df_2012.columns = ['index','index2', 'vessel and flag','ice class','ship operator','cargo','port of loading','port and date','entry to nsr','exit from nsr','time elapsed on nsr','avg speed','index3']
df_2012.columns

Index(['index', 'index2', 'vessel and flag', 'ice class', 'ship operator',
       'cargo', 'port of loading', 'port and date', 'entry to nsr',
       'exit from nsr', 'time elapsed on nsr', 'avg speed', 'index3'],
      dtype='object')

In [22]:
df_2012 = df_2012.drop(['index','index2','index3'],axis=1)
df_2012.columns

Index(['vessel and flag', 'ice class', 'ship operator', 'cargo',
       'port of loading', 'port and date', 'entry to nsr', 'exit from nsr',
       'time elapsed on nsr', 'avg speed'],
      dtype='object')

In [23]:
df_2012 = basic_clean(df_2012)
df_2012.head()

[ cargo 4 ]
[ port of loading 4 ]
[ port of loading 5 ]
[ avg speed 8 ]
[ avg speed 9 ]
[ avg speed 13 ]
[ cargo 30 ]
[ port of loading 38 ]
[ vessel and flag 41 ]
[ entry to nsr 45 ]
[ exit from nsr 45 ]
[ ship operator 46 ]
[ vessel and flag 48 ]
[ port of loading 50 ]
[ cargo 51 ]
New line characters have been removed. Detected 15 cell(s) with non-ASCII characters.
Null Count:
 vessel and flag        0
ice class              2
ship operator          3
cargo                  1
port of loading        2
port and date          1
entry to nsr           2
exit from nsr          4
time elapsed on nsr    8
avg speed              8
dtype: int64


,vessel and flag,ice class,ship operator,cargo,port of loading,port and date,entry to nsr,exit from nsr,time elapsed on nsr,avg speed
0,Vessel and Flag,Ice Class,Shipowner / Operator,Cargo,Destination,Port and Date of Sail,Entry to NSR,Exit from NSR,Time on NSR,Average speed. knots
1,"mt ""Varzuga"" Russia",Arc5,JSC MSCo,12538 t Diesel fuel,Pevek-Anadyr,Murmansk 20.06.12 22:30,23.06.12 06:00,16.07.12 10:00,To Pevek 15.4 days From Pevek to Dezhnev 4.1 days,5.8.
2,"Mt ""Indiga"" Russia",Arc5,JSC MSCo,12979 t Diesel Fuel,Pevek-Anadyr,Murmansk 21.06.12 22:00,24.06.12 05:00,16.07.12 10:00,To Pevek 14.3 days From Pevek to Dezhnev 4.1 days,5.9
3,"Supply vessel ""Vengeri"" Russia",Ice 1A Super (Arc 5),FEMCO LTD,repositioning. dspl 3733 t,K holmsk. Sakhalin,Murmansk 05.07.12 13:00,08.07.12 04:00,23.07.12 00:00,14.8,7.3
4,"mt ""Nordic Odyssey"" PANAMA",Ice 1A (Arc4),Nordic Bulk Carriers,"67520 t ЖРК ""Еврохим""",Китай,Murmansk 10.07.12 02:00,11.07.12 22:00,23.07.12 05:00,11.2,9.7


In [24]:
df_2012 = df_2012[1:]
df_2012.reset_index(inplace=True)
df_2012 = df_2012.drop('index',axis=1)
df_2012.head()

,vessel and flag,ice class,ship operator,cargo,port of loading,port and date,entry to nsr,exit from nsr,time elapsed on nsr,avg speed
0,"mt ""Varzuga"" Russia",Arc5,JSC MSCo,12538 t Diesel fuel,Pevek-Anadyr,Murmansk 20.06.12 22:30,23.06.12 06:00,16.07.12 10:00,To Pevek 15.4 days From Pevek to Dezhnev 4.1 days,5.8.
1,"Mt ""Indiga"" Russia",Arc5,JSC MSCo,12979 t Diesel Fuel,Pevek-Anadyr,Murmansk 21.06.12 22:00,24.06.12 05:00,16.07.12 10:00,To Pevek 14.3 days From Pevek to Dezhnev 4.1 days,5.9
2,"Supply vessel ""Vengeri"" Russia",Ice 1A Super (Arc 5),FEMCO LTD,repositioning. dspl 3733 t,K holmsk. Sakhalin,Murmansk 05.07.12 13:00,08.07.12 04:00,23.07.12 00:00,14.8,7.3
3,"mt ""Nordic Odyssey"" PANAMA",Ice 1A (Arc4),Nordic Bulk Carriers,"67520 t ЖРК ""Еврохим""",Китай,Murmansk 10.07.12 02:00,11.07.12 22:00,23.07.12 05:00,11.2,9.7
4,"mt ""Marilee"" NORWAY",Ice 1A (Arc4),MARINVEST,60505 t gascondensat e NOVATEK,Inchеon. Republic of Korea,Murmansk 10.07.12 05:00,11.07.12 16:00,22.07.12 23:40,11.3,9.6


In [25]:
# split some columns to sequester info nicer
# splitting vessel and flag --> vessel, flag
# splitting port and date --> starting port, trip start date

df_2012[['vessel', 'flag']] = df_2012['vessel and flag'].str.rsplit(n=1, expand=True)
df_2012.drop('vessel and flag',axis=1,inplace=True)

df_2012[['starting port','starting date']] = df_2012['port and date'].str.split(n=1,expand=True)
df_2012.drop('port and date',axis=1, inplace=True)

# df_2012[['cargo weight','cargo type']] = df_2012['cargo'].str.split('',n=1,regex=True)
# df_2012.drop('cargo',axis=1,inplace=True)

df_2012[['starting date','starting time']] = df_2012['starting date'].str.split(n=1,expand=True)

df_2012[['nsr entry date','nsr entry time']] = df_2012['entry to nsr'].str.split(n=1,expand=True)
df_2012.drop('entry to nsr',axis=1,inplace=True)

df_2012[['nsr exit date','nsr exit time']] = df_2012['exit from nsr'].str.split(n=1,expand=True)
df_2012.drop('exit from nsr',axis=1,inplace=True)

df_2012.head()

,ice class,ship operator,cargo,port of loading,time elapsed on nsr,avg speed,vessel,flag,starting port,starting date,starting time,nsr entry date,nsr entry time,nsr exit date,nsr exit time
0,Arc5,JSC MSCo,12538 t Diesel fuel,Pevek-Anadyr,To Pevek 15.4 days From Pevek to Dezhnev 4.1 days,5.8.,"mt ""Varzuga""",Russia,Murmansk,20.06.12,22:30,23.06.12,06:00,16.07.12,10:00
1,Arc5,JSC MSCo,12979 t Diesel Fuel,Pevek-Anadyr,To Pevek 14.3 days From Pevek to Dezhnev 4.1 days,5.9,"Mt ""Indiga""",Russia,Murmansk,21.06.12,22:00,24.06.12,05:00,16.07.12,10:00
2,Ice 1A Super (Arc 5),FEMCO LTD,repositioning. dspl 3733 t,K holmsk. Sakhalin,14.8,7.3,"Supply vessel ""Vengeri""",Russia,Murmansk,05.07.12,13:00,08.07.12,04:00,23.07.12,00:00
3,Ice 1A (Arc4),Nordic Bulk Carriers,"67520 t ЖРК ""Еврохим""",Китай,11.2,9.7,"mt ""Nordic Odyssey""",PANAMA,Murmansk,10.07.12,02:00,11.07.12,22:00,23.07.12,05:00
4,Ice 1A (Arc4),MARINVEST,60505 t gascondensat e NOVATEK,Inchеon. Republic of Korea,11.3,9.6,"mt ""Marilee""",NORWAY,Murmansk,10.07.12,05:00,11.07.12,16:00,22.07.12,23:40


In [26]:
# fix columns that have inconsistent formatting/ additional information

df_2012['avg speed'] = df_2012['avg speed'].str.rstrip('. уз')

# regex matches and creates a consistent format for time 
# allowing for change to column dtype
df_2012['starting time'] = df_2012['starting time'].str.extract(r'((?:[01]\d|2[0-3]):[0-5]\d)',expand=False)
df_2012['nsr entry time'] = df_2012['nsr entry time'].str.extract(r'((?:[01]\d|2[0-3]):[0-5]\d)',expand=False)
df_2012['nsr exit time'] = df_2012['nsr exit time'].str.extract(r'((?:[01]\d|2[0-3]):[0-5]\d)',expand=False)

# regex matches all digits, accounting for whitespace, in a single capture group
df_2012['cargo mass'] = df_2012['cargo'].str.extract(r'((?:\d+\s*)+)')


In [27]:
df_2012.head()

,ice class,ship operator,cargo,port of loading,time elapsed on nsr,avg speed,vessel,flag,starting port,starting date,starting time,nsr entry date,nsr entry time,nsr exit date,nsr exit time,cargo mass
0,Arc5,JSC MSCo,12538 t Diesel fuel,Pevek-Anadyr,To Pevek 15.4 days From Pevek to Dezhnev 4.1 days,5.8,"mt ""Varzuga""",Russia,Murmansk,20.06.12,22:30,23.06.12,06:00,16.07.12,10:00,12538
1,Arc5,JSC MSCo,12979 t Diesel Fuel,Pevek-Anadyr,To Pevek 14.3 days From Pevek to Dezhnev 4.1 days,5.9,"Mt ""Indiga""",Russia,Murmansk,21.06.12,22:00,24.06.12,05:00,16.07.12,10:00,12979
2,Ice 1A Super (Arc 5),FEMCO LTD,repositioning. dspl 3733 t,K holmsk. Sakhalin,14.8,7.3,"Supply vessel ""Vengeri""",Russia,Murmansk,05.07.12,13:00,08.07.12,04:00,23.07.12,00:00,3733
3,Ice 1A (Arc4),Nordic Bulk Carriers,"67520 t ЖРК ""Еврохим""",Китай,11.2,9.7,"mt ""Nordic Odyssey""",PANAMA,Murmansk,10.07.12,02:00,11.07.12,22:00,23.07.12,05:00,67520
4,Ice 1A (Arc4),MARINVEST,60505 t gascondensat e NOVATEK,Inchеon. Republic of Korea,11.3,9.6,"mt ""Marilee""",NORWAY,Murmansk,10.07.12,05:00,11.07.12,16:00,22.07.12,23:40,60505


In [28]:
# outlier cells need to conform

df_2012.loc[0,'time elapsed on nsr'] = 19.5
df_2012.loc[1,'time elapsed on nsr'] = 18.4

In [29]:
# deal with excess table headings that carried over from PDF version

df_2012.drop(9,axis=0,inplace=True)
df_2012.drop(10,axis=0,inplace=True)

df_2012.loc[8,'cargo'] = '60310 t. gas condensate NOVATEK'
df_2012.loc[8,'vessel'] = 'Mt Palva'
df_2012.loc[8,'flag'] = 'FINLAND'
df_2012.loc[8,'starting time'] = '19:30'


In [30]:
df_2012.drop(20,axis=0,inplace=True)
df_2012.drop(21,axis=0,inplace=True)

df_2012.loc[19,'vessel'] = 'supply vessel Tumcha (towing pipe-layer Nebula)'
df_2012.loc[19,'flag'] = 'Russia'
df_2012.loc[19,'starting port'] = 'Petropavlovsk-Kamchatskiy'
df_2012.loc[19,'starting date'] = '22.08.12'
df_2012.loc[19,'starting time'] = '14:10'


In [31]:
df_2012.drop(32,axis=0,inplace=True)
df_2012.drop(33,axis=0,inplace=True)

df_2012.loc[31,'cargo'] = '66275 t. Jet Fuel'
df_2012.loc[31,'port of loading'] = 'Porvoo, Finland'
df_2012.loc[31,'flag'] = 'Finland'
df_2012.loc[31,'vessel'] = 'mt Palva'
df_2012.loc[31,'starting port'] = 'Yosu, Republic of Korea'
df_2012.loc[31,'starting date'] = '05.09.12'
df_2012.loc[31,'nsr entry time'] = '10:35'
df_2012.loc[31,'nsr exit time'] = '21:30'


In [32]:
# clean up both cargo and cargo mass columns

df_2012['cargo mass'] = df_2012['cargo mass'].str.replace('\n','')

# create lambda function to run r split or l split based off of first character
df_2012['cargo'] = df_2012['cargo'].apply(lambda c : c.split(maxsplit=1) if str(c)[0].isnumeric() else c.rsplit(maxsplit=1))

In [33]:
# drop last superfluous header

df_2012.drop(43,axis=0,inplace=True)


In [34]:
# fix rows that are out of standard
# clear theme stands out as limitation of pdfplumber and original PDFs: 
#     multiple types of data within a single cell creates problems

df_2012.loc[0,'avg speed'] = 5.8

df_2012.loc[3,'port of loading'] = 'China'

df_2012.loc[6,'starting port'] = 'Yosu, Republic of Korea'
df_2012.loc[6,'starting date'] = '30.06.12'
df_2012.loc[6,'starting time'] = '02:15'

df_2012.loc[7,'starting port'] = 'Shanghai, China'
df_2012.loc[7,'starting date'] = np.nan
df_2012.loc[7,'starting time'] = np.nan

df_2012.loc[11,'starting port'] = 'Port Vostchnyi'
df_2012.loc[11,'starting date'] = '28.07.12'
df_2012.loc[11,'starting time'] = np.nan

df_2012.loc[13,'starting port'] = 'Port of Nakhodka'
df_2012.loc[13,'starting date'] = '05.08.12'
df_2012.loc[13,'starting time'] = '14:30'

df_2012.loc[14,'vessel'] = 'Mt "Yaroslav Mudryi"'
df_2012.loc[14,'flag'] = np.nan

df_2012.loc[15,'starting port'] = 'Huanghua, China'
df_2012.loc[15,'starting date'] = '11.08.12'
df_2012.loc[15,'starting time'] = '11:00'

df_2012.loc[16,'starting port'] = 'Yosu, Republic of Korea'
df_2012.loc[16,'starting date'] = '11.08.12'
df_2012.loc[16,'starting time'] = '13:00'

df_2012.loc[17,'vessel'] = 'mt STI Harmony'
df_2012.loc[17,'flag'] = 'Marshall Islands'

df_2012.loc[22,'starting port'] = 'Petropavlovsk-Kamchatskiy'
df_2012.loc[22,'starting date'] = '28.08.12'
df_2012.loc[22,'starting time'] = '17:00'

df_2012.loc[29,'starting port'] = 'Huanghua, China'
df_2012.loc[29,'starting date'] = '06.09.12'
df_2012.loc[29,'starting time'] = '14:00'

df_2012.loc[31,'starting time'] = np.nan

df_2012.loc[35,'starting time'] = np.nan

df_2012.loc[36,'starting port'] = 'Baydaratskaya Bay'
df_2012.loc[36,'starting date'] = '19.09.12'
df_2012.loc[36,'starting time'] = '11:00'
df_2012.loc[36,'nsr entry date'] = np.nan
df_2012.loc[36,'nsr entry time'] = np.nan

df_2012.loc[38,'starting port'] = 'Baydaratskaya Bay'
df_2012.loc[38,'starting date'] = '01.10.12'
df_2012.loc[38,'starting time'] = '17:00'

df_2012.loc[40,'vessel'] = 'LNG Ob River'
df_2012.loc[40,'flag'] = 'Marshall Islands'
df_2012.loc[40,'starting port'] = 'Yosu, Republic of Korea'
df_2012.loc[40,'starting date'] = '30.09.12'
df_2012.loc[40,'starting time'] = '01:00'

df_2012.loc[42,'starting time'] = np.nan

df_2012.loc[46,'starting port'] = 'Qinhuangdao, China'
df_2012.loc[46,'starting date'] = np.nan
df_2012.loc[46,'starting time'] = np.nan

df_2012.loc[47,'vessel'] = 'Slavyanka Bay'
df_2012.loc[47,'flag'] = 'Russia'
df_2012.loc[47,'starting port'] = 'Petropavlovsk-Kamchatskiy'
df_2012.loc[47,'starting date'] = np.nan
df_2012.loc[47,'starting time'] = np.nan

df_2012.loc[48,'starting port'] = 'Vancouver, Canada'
df_2012.loc[48,'starting date'] = '26.10.12'
df_2012.loc[48,'starting time'] = '04:00'

df_2012.loc[49,'port of loading'] = 'Murmansk'
df_2012.loc[49,'starting port'] = 'Caofeidian, China'
df_2012.loc[49,'starting date'] = '28.10.12'
df_2012.loc[49,'starting time'] = '04:00'

df_2012.loc[50,'vessel'] = 'LNG Ob River'
df_2012.loc[50,'flag'] = 'Marshall Islands'
df_2012.loc[50,'starting port'] = 'Hammerfest, Norway'
df_2012.loc[50,'starting date'] = '07.11.12'
df_2012.loc[50,'starting time'] = '18:55'

df_2012.loc[51,'starting port'] = 'Alaska, USA'
df_2012.loc[51,'starting date'] = '12.11.12'
df_2012.loc[51,'starting time'] = '08:30'

df_2012.loc[52,'starting port'] = 'Alaska, USA'
df_2012.loc[52,'starting date'] = '12.11.12'
df_2012.loc[52,'starting time'] = '07:45'


df_2012

,ice class,ship operator,cargo,port of loading,time elapsed on nsr,avg speed,vessel,flag,starting port,starting date,starting time,nsr entry date,nsr entry time,nsr exit date,nsr exit time,cargo mass
0,Arc5,JSC MSCo,"[12538, t Diesel fuel]",Pevek-Anadyr,19.5,5.8,"mt ""Varzuga""",Russia,Murmansk,20.06.12,22:30,23.06.12,06:00,16.07.12,10:00,12538
1,Arc5,JSC MSCo,"[12979, t Diesel Fuel]",Pevek-Anadyr,18.4,5.9,"Mt ""Indiga""",Russia,Murmansk,21.06.12,22:00,24.06.12,05:00,16.07.12,10:00,12979
2,Ice 1A Super (Arc 5),FEMCO LTD,"[repositioning. dspl 3733, t]",K holmsk. Sakhalin,14.8,7.3,"Supply vessel ""Vengeri""",Russia,Murmansk,05.07.12,13:00,08.07.12,04:00,23.07.12,00:00,3733
3,Ice 1A (Arc4),Nordic Bulk Carriers,"[67520, t ЖРК ""Еврохим""]",China,11.2,9.7,"mt ""Nordic Odyssey""",PANAMA,Murmansk,10.07.12,02:00,11.07.12,22:00,23.07.12,05:00,67520
4,Ice 1A (Arc4),MARINVEST,"[60505, t gascondensat e NOVATEK]",Inchеon. Republic of Korea,11.3,9.6,"mt ""Marilee""",NORWAY,Murmansk,10.07.12,05:00,11.07.12,16:00,22.07.12,23:40,60505
5,Ice-1A (Arc4),Nordic Bulk Carriers,"[ballast dspl 87216, t.]",Murmansk,10.8,10.0,"mv ""Nordic Orion""",PANAMA,Shanghai,10.07.12,07:35,20.07.12,19:45,31.07.12,15:20,87216
6,Ice 1A (Arc4),Neste Oil,"[66416, t Jet fuel]",Porvoo. Finland,11.5,9.4,"mt ""Stena Poseidon""",FINLAND,"Yosu, Republic of Korea",30.06.12,02:15,20.07.12,09:00,31.07.12,20:50,66416
7,Arc 5,Polar Research Institute of China,"[repositioning dspl. 21025, t.]",Iceland,9.9,10.9,"icebreaker ""Xuelong""",CHINA,"Shanghai, China",NaN,NaN,21.07.12,17:00,31.07.12,15:00,21025
8,Ice 1A (Arc4),"""Neste Oil""","[60310, t. gas condensate NOVATEK]",Daesan. Republic of Korea,8.8,12.3,Mt Palva,FINLAND,Murmansk,23.07.12,19:30,26.07.12,NaN,04.08.12,11:50,60310
11,Arc 5,Partner LTD,"[6500, t. Diesel fuel]",Kolyma. Arkhangelsk,NaN,NaN,"mt ""Kasla""",Russia,Port Vostchnyi,28.07.12,NaN,09.08.12,10:00,02.09.12,02:00,6500


In [35]:
# cargo was split up for 'cargo mass' 
# need to pull out only type of cargo 
# cells will be in list form 
#     if cargo is for ballast then first item will not begin with a number

for index, cell in df_2012['cargo'].items():
    if not cell[0][0].isnumeric():
        df_2012.loc[index, 'cargo'] = 'ballast'
    else:
        df_2012.loc[index,'cargo'] = cell[1]

# remove leftover ton units

df_2012['cargo'] = df_2012['cargo'].str.split(n=1)

df_2012['cargo'] = [ cell[-1] for index, cell in df_2012['cargo'].items()]
    
df_2012.loc[3,'cargo'] = 'iron ore concentrate'


In [36]:
df_2012.info()

<class 'pandas.core.frame.DataFrame'>
Index: 46 entries, 0 to 52
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ice class            46 non-null     object
 1   ship operator        46 non-null     object
 2   cargo                46 non-null     object
 3   port of loading      46 non-null     object
 4   time elapsed on nsr  41 non-null     object
 5   avg speed            41 non-null     object
 6   vessel               46 non-null     object
 7   flag                 45 non-null     object
 8   starting port        46 non-null     object
 9   starting date        43 non-null     object
 10  starting time        39 non-null     object
 11  nsr entry date       45 non-null     object
 12  nsr entry time       43 non-null     object
 13  nsr exit date        44 non-null     object
 14  nsr exit time        42 non-null     object
 15  cargo mass           46 non-null     object
dtypes: object(16)
m

In [37]:
# convert columns to more specific data types

df_2012['time elapsed on nsr'] = pd.to_numeric(df_2012['time elapsed on nsr'])
df_2012['avg speed'] = pd.to_numeric(df_2012['avg speed'])

df_2012['cargo mass'] = df_2012['cargo mass'].str.replace(" ","",regex=True)
df_2012['cargo mass'] = df_2012['cargo mass'].str.replace("\n","",regex=True)
df_2012['cargo mass'] = pd.to_numeric(df_2012['cargo mass'])
df_2012.drop('cargo mass',axis=1,inplace=True)

df_2012['starting date'] = pd.to_datetime(df_2012['starting date'],format='%d.%m.%y')
df_2012['nsr entry date'] = pd.to_datetime(df_2012['nsr entry date'],format='%d.%m.%y')
df_2012['nsr exit date'] = pd.to_datetime(df_2012['nsr exit date'],format='%d.%m.%y')

df_2012['starting time'] = pd.to_datetime(df_2012['starting time'],format='%H:%M')
df_2012['nsr entry time'] = pd.to_datetime(df_2012['nsr entry time'],format='%H:%M')
df_2012['nsr exit time'] = pd.to_datetime(df_2012['nsr exit time'],format='%H:%M')

df_2012['starting time'] = df_2012['starting time'].dt.strftime('%H:%M')
df_2012['nsr entry time'] = df_2012['nsr entry time'].dt.strftime('%H:%M')
df_2012['nsr exit time'] = df_2012['nsr exit time'].dt.strftime('%H:%M')

df_2012[['ice class','ship operator','cargo','port of loading','vessel','flag','starting port']] = df_2012[['ice class','ship operator','cargo','port of loading','vessel','flag','starting port']].astype('str')

df_2012.info()

<class 'pandas.core.frame.DataFrame'>
Index: 46 entries, 0 to 52
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ice class            46 non-null     object        
 1   ship operator        46 non-null     object        
 2   cargo                46 non-null     object        
 3   port of loading      46 non-null     object        
 4   time elapsed on nsr  41 non-null     float64       
 5   avg speed            41 non-null     float64       
 6   vessel               46 non-null     object        
 7   flag                 46 non-null     object        
 8   starting port        46 non-null     object        
 9   starting date        43 non-null     datetime64[ns]
 10  starting time        39 non-null     object        
 11  nsr entry date       45 non-null     datetime64[ns]
 12  nsr entry time       43 non-null     object        
 13  nsr exit date        44 non-null     datet

In [38]:
df_2012.columns = [
    'ice class','ship operator','cargo type','port of arrival','time elapsed on nsr','avg speed',
    'vessel name','flag','port of departure','date of departure','time of departure','nsr entry date',
    'nsr entry time','nsr exit date','nsr exit time'
]

In [39]:
df_2012.head()

,ice class,ship operator,cargo type,port of arrival,time elapsed on nsr,avg speed,vessel name,flag,port of departure,date of departure,time of departure,nsr entry date,nsr entry time,nsr exit date,nsr exit time
0,Arc5,JSC MSCo,Diesel fuel,Pevek-Anadyr,19.5,5.8,"mt ""Varzuga""",Russia,Murmansk,2012-06-20,22:30,2012-06-23,06:00,2012-07-16,10:00
1,Arc5,JSC MSCo,Diesel Fuel,Pevek-Anadyr,18.4,5.9,"Mt ""Indiga""",Russia,Murmansk,2012-06-21,22:00,2012-06-24,05:00,2012-07-16,10:00
2,Ice 1A Super (Arc 5),FEMCO LTD,ballast,K holmsk. Sakhalin,14.8,7.3,"Supply vessel ""Vengeri""",Russia,Murmansk,2012-07-05,13:00,2012-07-08,04:00,2012-07-23,00:00
3,Ice 1A (Arc4),Nordic Bulk Carriers,iron ore concentrate,China,11.2,9.7,"mt ""Nordic Odyssey""",PANAMA,Murmansk,2012-07-10,02:00,2012-07-11,22:00,2012-07-23,05:00
4,Ice 1A (Arc4),MARINVEST,gascondensat e NOVATEK,Inchеon. Republic of Korea,11.3,9.6,"mt ""Marilee""",NORWAY,Murmansk,2012-07-10,05:00,2012-07-11,16:00,2012-07-22,23:40


In [40]:
# 2013

df_2013 = data[2013]
df_2013.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0,No.,Vessel and Flag,Ice class,"GRT,\ntones",Vessel\nowner/operator,Cargo,Port of\ndestination,Port and date\nof departure,Entry to\nNSR,Exit from\nNSR,Days\nspent at\nNSR,"Average\nspeed,\nknots",NaN,NaN,NaN
1,1,1,mt\n«Varzuga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,13 658 tones\ndiesel fuel,Певек -\nNakhodka,Murmansk\n25.06.13\n00:00,28.06.13\n16:00\nc. Zhelaniya,14.07.13\n14:00\nc. Dezhnev,"15,3","7,0",NaN,NaN,NaN
2,2,2,mt\n«Indiga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,13 645 tones\ndiesel fuel,Pevek,Murmansk\n25.06.13\n19:00,28.06.13\n17:00\nc. Zhelaniya,10.07.13\n09:20\nPevek,"11,7","9,2",NaN,NaN,NaN
3,3,3,mv\n«Nordic Orion»\nflag PANAMA,Ice-1A\n(Arc4),40 142,Nordic Bulk\nCarriers,66 000 tones\niron ore\nEurochem,"Lanshan, China",Murmansk\n01.07.13\n01:30,03.07.13\n10:00\nc. Zhelaniya,23.07.13\n08:55\nc. Dezhnev,"19,9","5,4",NaN,NaN,NaN
4,4,4,mv\n«Boris Vilkitskiy»\nflag RUSSIA,Arc4,5 025,Khatanga Sea\nMerchant Port\nCJSC,5 996 tones\ndiesel fuel,Khatanga Bay,NaN,13.07.13\n12:00\nYugorski\nShar,07.08.13\n24:00\nKhatanga Bay,"25,2","4,3",NaN,NaN,NaN


In [41]:
df_2013.tail(10)

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
75,75,67,mv «HHL Lagos»\nflag Antigua and\nBarbuda,Ice 1A\n(Arc4),17 644,Hansa Heavylift,3 000 tones\ngeneral cargo,Vladivostok,СПб\n25.10.13\n23:00,03.11.13\n14:00\nc. Zhelaniya,17.11.13\n09:30\nc. Dezhnev,"13,8","10,1",NaN,NaN,NaN
76,76,68,mv «Yuri\nArshenevsky»,Arc7,18 574,MSCO,"ballasting,\nFD 33 944",Arkhangelsk,Pevek\n06.11.13,06.11.13\n16:00,17.11.13\n07:40,"10,7","8,2",NaN,NaN,NaN
77,77,Type of\ncargo,Amount\nof\nvessels,"Cargo\nvolume,\ntones","Full displ.\n(FD), tones","GRT,\ntones",Laden\nvessels\neastbound\n(EB),"EB\ncargo\nvolume,\ntones","EB\nvessels\nGRT,\ntones",Laden\nvessels\nwestbound\n(WB),"WB\ncargo\nvolume,\ntones","WB\nvessels\nGRT,\ntones",Empty\nvessels\nEB,"EB\ndispl.,\ntones",Empty\nvessels\nWB,"WB\ndispl.,\ntones"
78,78,Liquid,31,911 867,NaN,666 956,23,588 659,431 321,8,323 208,235 635,NaN,NaN,NaN,NaN
79,79,Bulk,4,276 939,NaN,161 398,3,203 439,121 256,1,73 500,40 142,NaN,NaN,NaN,NaN
80,80,LNG,1,66 868,NaN,102 236,1,66 868,102 236,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,81,General cargo,13,100 223,NaN,152 378,9,36 846,102 206,4,63 377,50 172,NaN,NaN,NaN,NaN
82,82,Ballasting,15,NaN,469 703,285 710,NaN,NaN,1 579,NaN,NaN,284 131,1,2 258,14,467 445
83,83,Reposition,7,NaN,38 027,28 373,NaN,NaN,7 917,NaN,NaN,20 456,4,8 639,3,29 388
84,84,TOTAL:,71,1 355 897,507 730,1 397 051,36,895 812,766 515,13,460 085,630 536,5,10 897,17,496 833


In [42]:
df_2013_totals = df_2013[77:85]
df_2013_totals.head(10)

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
77,77,Type of\ncargo,Amount\nof\nvessels,"Cargo\nvolume,\ntones","Full displ.\n(FD), tones","GRT,\ntones",Laden\nvessels\neastbound\n(EB),"EB\ncargo\nvolume,\ntones","EB\nvessels\nGRT,\ntones",Laden\nvessels\nwestbound\n(WB),"WB\ncargo\nvolume,\ntones","WB\nvessels\nGRT,\ntones",Empty\nvessels\nEB,"EB\ndispl.,\ntones",Empty\nvessels\nWB,"WB\ndispl.,\ntones"
78,78,Liquid,31,911 867,NaN,666 956,23,588 659,431 321,8,323 208,235 635,NaN,NaN,NaN,NaN
79,79,Bulk,4,276 939,NaN,161 398,3,203 439,121 256,1,73 500,40 142,NaN,NaN,NaN,NaN
80,80,LNG,1,66 868,NaN,102 236,1,66 868,102 236,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,81,General cargo,13,100 223,NaN,152 378,9,36 846,102 206,4,63 377,50 172,NaN,NaN,NaN,NaN
82,82,Ballasting,15,NaN,469 703,285 710,NaN,NaN,1 579,NaN,NaN,284 131,1,2 258,14,467 445
83,83,Reposition,7,NaN,38 027,28 373,NaN,NaN,7 917,NaN,NaN,20 456,4,8 639,3,29 388
84,84,TOTAL:,71,1 355 897,507 730,1 397 051,36,895 812,766 515,13,460 085,630 536,5,10 897,17,496 833


In [43]:
df_2013 = df_2013[:77]
df_2013.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
72,72,64,mv «HHL Hong\nKong»\nflag LIBERIA,Ice 1A\n(Arc4),17 634,Hansa Heavylift,1 742 tones\ngeneral cargo,Rajin\n(N. Kores),"Ust-Luga,\n08.10.13",25.10.13\n09:30\nKara Gate,11.11.13\n02:40\nc. Dezhnev,"16,7","6,7",NaN,NaN,NaN
73,73,65,icebreaker\n«Fennica»\nflag FINLAND,Polar-10\nIcebreaker\n(DNV),9 392,Arctia Offshore\nLtd.,"reposition,\nFD\n12 800 tones","Kirkenes,\nNorway","Dutch Harbor,\nUSA\n28.10.13",31.10.13\n01:45\nc. Dezhnev,18.11.13\n20:00\nKara Gate,"18,8","7,6",NaN,NaN,NaN
74,74,66,mt\n«Indiga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,9 436 tones\ndiesel fuel,Pevek,Murmansk\n30.10.13\n08:10,02.11.13\n08:15\nc. Zhelaniya,15.11.13\n10:00\nPevek,"13,1","10,6",NaN,NaN,NaN
75,75,67,mv «HHL Lagos»\nflag Antigua and\nBarbuda,Ice 1A\n(Arc4),17 644,Hansa Heavylift,3 000 tones\ngeneral cargo,Vladivostok,СПб\n25.10.13\n23:00,03.11.13\n14:00\nc. Zhelaniya,17.11.13\n09:30\nc. Dezhnev,"13,8","10,1",NaN,NaN,NaN
76,76,68,mv «Yuri\nArshenevsky»,Arc7,18 574,MSCO,"ballasting,\nFD 33 944",Arkhangelsk,Pevek\n06.11.13,06.11.13\n16:00,17.11.13\n07:40,"10,7","8,2",NaN,NaN,NaN


In [44]:
df_2013.columns = ['index','index2','vessel and flag','ice class','grt (tons)','vessel operator','cargo','port of destination','port and date of departure','entry to nsr','exit from nsr','time elapsed on nsr','avg speed (knots)','index3','index4','index5']
df_2013.head()

,index,index2,vessel and flag,ice class,grt (tons),vessel operator,cargo,port of destination,port and date of departure,entry to nsr,exit from nsr,time elapsed on nsr,avg speed (knots),index3,index4,index5
0,0,No.,Vessel and Flag,Ice class,"GRT,\ntones",Vessel\nowner/operator,Cargo,Port of\ndestination,Port and date\nof departure,Entry to\nNSR,Exit from\nNSR,Days\nspent at\nNSR,"Average\nspeed,\nknots",NaN,NaN,NaN
1,1,1,mt\n«Varzuga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,13 658 tones\ndiesel fuel,Певек -\nNakhodka,Murmansk\n25.06.13\n00:00,28.06.13\n16:00\nc. Zhelaniya,14.07.13\n14:00\nc. Dezhnev,"15,3","7,0",NaN,NaN,NaN
2,2,2,mt\n«Indiga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,13 645 tones\ndiesel fuel,Pevek,Murmansk\n25.06.13\n19:00,28.06.13\n17:00\nc. Zhelaniya,10.07.13\n09:20\nPevek,"11,7","9,2",NaN,NaN,NaN
3,3,3,mv\n«Nordic Orion»\nflag PANAMA,Ice-1A\n(Arc4),40 142,Nordic Bulk\nCarriers,66 000 tones\niron ore\nEurochem,"Lanshan, China",Murmansk\n01.07.13\n01:30,03.07.13\n10:00\nc. Zhelaniya,23.07.13\n08:55\nc. Dezhnev,"19,9","5,4",NaN,NaN,NaN
4,4,4,mv\n«Boris Vilkitskiy»\nflag RUSSIA,Arc4,5 025,Khatanga Sea\nMerchant Port\nCJSC,5 996 tones\ndiesel fuel,Khatanga Bay,NaN,13.07.13\n12:00\nYugorski\nShar,07.08.13\n24:00\nKhatanga Bay,"25,2","4,3",NaN,NaN,NaN


In [45]:
df_2013 = df_2013.drop(['index','index2','index3','index4','index5'],axis=1)
df_2013 = df_2013.drop([0],axis=0)
df_2013.head()

,vessel and flag,ice class,grt (tons),vessel operator,cargo,port of destination,port and date of departure,entry to nsr,exit from nsr,time elapsed on nsr,avg speed (knots)
1,mt\n«Varzuga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,13 658 tones\ndiesel fuel,Певек -\nNakhodka,Murmansk\n25.06.13\n00:00,28.06.13\n16:00\nc. Zhelaniya,14.07.13\n14:00\nc. Dezhnev,"15,3","7,0"
2,mt\n«Indiga»\nflag RUSSIA,Arc5,11 290,MSCO /\nChukotsnab,13 645 tones\ndiesel fuel,Pevek,Murmansk\n25.06.13\n19:00,28.06.13\n17:00\nc. Zhelaniya,10.07.13\n09:20\nPevek,"11,7","9,2"
3,mv\n«Nordic Orion»\nflag PANAMA,Ice-1A\n(Arc4),40 142,Nordic Bulk\nCarriers,66 000 tones\niron ore\nEurochem,"Lanshan, China",Murmansk\n01.07.13\n01:30,03.07.13\n10:00\nc. Zhelaniya,23.07.13\n08:55\nc. Dezhnev,"19,9","5,4"
4,mv\n«Boris Vilkitskiy»\nflag RUSSIA,Arc4,5 025,Khatanga Sea\nMerchant Port\nCJSC,5 996 tones\ndiesel fuel,Khatanga Bay,NaN,13.07.13\n12:00\nYugorski\nShar,07.08.13\n24:00\nKhatanga Bay,"25,2","4,3"
5,mt\n«Indiga»\nflag RUSSIA,Arc5,11 290,MSCO/\nChukotsnab,"ballasting,\nFD 22 654\ntones",Ob Bay,Pevek\n15.07.13\n01:30,15.07.13\n01:30\nPevek,24.07.13\n12:50\nOb Bay,"11,3","7,2"


In [46]:
df_2013 = basic_clean(df_2013)

[ port of destination 1 ]
[ vessel and flag 10 ]
[ cargo 34 ]
[ port and date of departure 75 ]
New line characters have been removed. Detected 4 cell(s) with non-ASCII characters.
Null Count:
 vessel and flag               1
ice class                     2
grt (tons)                    3
vessel operator               2
cargo                         1
port of destination           1
port and date of departure    1
entry to nsr                  1
exit from nsr                 1
time elapsed on nsr           3
avg speed (knots)             3
dtype: int64


In [47]:
df_2013.head()

,vessel and flag,ice class,grt (tons),vessel operator,cargo,port of destination,port and date of departure,entry to nsr,exit from nsr,time elapsed on nsr,avg speed (knots)
1,"mt ""Varzuga"" flag RUSSIA",Arc5,11 290,MSCO / Chukotsnab,13 658 tones diesel fuel,Певек - Nakhodka,Murmansk 25.06.13 00:00,28.06.13 16:00 c. Zhelaniya,14.07.13 14:00 c. Dezhnev,15.3,7.0
2,"mt ""Indiga"" flag RUSSIA",Arc5,11 290,MSCO / Chukotsnab,13 645 tones diesel fuel,Pevek,Murmansk 25.06.13 19:00,28.06.13 17:00 c. Zhelaniya,10.07.13 09:20 Pevek,11.7,9.2
3,"mv ""Nordic Orion"" flag PANAMA",Ice-1A (Arc4),40 142,Nordic Bulk Carriers,66 000 tones iron ore Eurochem,Lanshan. China,Murmansk 01.07.13 01:30,03.07.13 10:00 c. Zhelaniya,23.07.13 08:55 c. Dezhnev,19.9,5.4
4,"mv ""Boris Vilkitskiy"" flag RUSSIA",Arc4,5 025,Khatanga Sea Merchant Port CJSC,5 996 tones diesel fuel,Khatanga Bay,NaN,13.07.13 12:00 Yugorski Shar,07.08.13 24:00 Khatanga Bay,25.2,4.3
5,"mt ""Indiga"" flag RUSSIA",Arc5,11 290,MSCO/ Chukotsnab,ballasting. FD 22 654 tones,Ob Bay,Pevek 15.07.13 01:30,15.07.13 01:30 Pevek,24.07.13 12:50 Ob Bay,11.3,7.2


In [48]:
# split columns into standardized column format

df_2013[['vessel','flag']] = df_2013['vessel and flag'].str.split('flag',n=1,expand=True)
df_2013.drop('vessel and flag', axis=1,inplace=True)

df_2013[['starting port','starting date','starting time']] = df_2013['port and date of departure'].str.split(n=2,expand=True)
df_2013.drop('port and date of departure',axis=1,inplace=True)

df_2013[['nsr entry date','nsr entry time','nsr entry location']] = df_2013['entry to nsr'].str.split(n=2,expand=True)
df_2013.drop('entry to nsr',axis=1,inplace=True)

df_2013[['nsr exit date','nsr exit time','nsr exit location']] = df_2013['exit from nsr'].str.split(n=2,expand=True)
df_2013.drop('exit from nsr',axis=1,inplace=True)

df_2013.head()

,ice class,grt (tons),vessel operator,cargo,port of destination,time elapsed on nsr,avg speed (knots),vessel,flag,starting port,starting date,starting time,nsr entry date,nsr entry time,nsr entry location,nsr exit date,nsr exit time,nsr exit location
1,Arc5,11 290,MSCO / Chukotsnab,13 658 tones diesel fuel,Певек - Nakhodka,15.3,7.0,"mt ""Varzuga""",RUSSIA,Murmansk,25.06.13,00:00,28.06.13,16:00,c. Zhelaniya,14.07.13,14:00,c. Dezhnev
2,Arc5,11 290,MSCO / Chukotsnab,13 645 tones diesel fuel,Pevek,11.7,9.2,"mt ""Indiga""",RUSSIA,Murmansk,25.06.13,19:00,28.06.13,17:00,c. Zhelaniya,10.07.13,09:20,Pevek
3,Ice-1A (Arc4),40 142,Nordic Bulk Carriers,66 000 tones iron ore Eurochem,Lanshan. China,19.9,5.4,"mv ""Nordic Orion""",PANAMA,Murmansk,01.07.13,01:30,03.07.13,10:00,c. Zhelaniya,23.07.13,08:55,c. Dezhnev
4,Arc4,5 025,Khatanga Sea Merchant Port CJSC,5 996 tones diesel fuel,Khatanga Bay,25.2,4.3,"mv ""Boris Vilkitskiy""",RUSSIA,NaN,NaN,NaN,13.07.13,12:00,Yugorski Shar,07.08.13,24:00,Khatanga Bay
5,Arc5,11 290,MSCO/ Chukotsnab,ballasting. FD 22 654 tones,Ob Bay,11.3,7.2,"mt ""Indiga""",RUSSIA,Pevek,15.07.13,01:30,15.07.13,01:30,Pevek,24.07.13,12:50,Ob Bay


In [49]:
# split cargo into cargo and cargo mass

# create lambda function to run r split or l split based off of first character
df_2013['cargo'] = df_2013['cargo'].apply(lambda c : str(c).rsplit('tones',maxsplit=1) if str(c)[0].isnumeric() else str(c).split(maxsplit=1))

for index, cell in df_2013['cargo'].items():
    if not cell[0][0].isnumeric():
        df_2013.loc[index, 'cargo'] = 'ballast'
        df_2013.loc[index,'cargo mass'] = cell[-1]
    else:
        df_2013.loc[index,'cargo'] = cell[-1]
        df_2013.loc[index,'cargo mass'] = cell[0]

In [50]:
df_2013.head()

,ice class,grt (tons),vessel operator,cargo,port of destination,time elapsed on nsr,avg speed (knots),vessel,flag,starting port,starting date,starting time,nsr entry date,nsr entry time,nsr entry location,nsr exit date,nsr exit time,nsr exit location,cargo mass
1,Arc5,11 290,MSCO / Chukotsnab,diesel fuel,Певек - Nakhodka,15.3,7.0,"mt ""Varzuga""",RUSSIA,Murmansk,25.06.13,00:00,28.06.13,16:00,c. Zhelaniya,14.07.13,14:00,c. Dezhnev,13 658
2,Arc5,11 290,MSCO / Chukotsnab,diesel fuel,Pevek,11.7,9.2,"mt ""Indiga""",RUSSIA,Murmansk,25.06.13,19:00,28.06.13,17:00,c. Zhelaniya,10.07.13,09:20,Pevek,13 645
3,Ice-1A (Arc4),40 142,Nordic Bulk Carriers,iron ore Eurochem,Lanshan. China,19.9,5.4,"mv ""Nordic Orion""",PANAMA,Murmansk,01.07.13,01:30,03.07.13,10:00,c. Zhelaniya,23.07.13,08:55,c. Dezhnev,66 000
4,Arc4,5 025,Khatanga Sea Merchant Port CJSC,diesel fuel,Khatanga Bay,25.2,4.3,"mv ""Boris Vilkitskiy""",RUSSIA,NaN,NaN,NaN,13.07.13,12:00,Yugorski Shar,07.08.13,24:00,Khatanga Bay,5 996
5,Arc5,11 290,MSCO/ Chukotsnab,ballast,Ob Bay,11.3,7.2,"mt ""Indiga""",RUSSIA,Pevek,15.07.13,01:30,15.07.13,01:30,Pevek,24.07.13,12:50,Ob Bay,FD 22 654 tones


In [51]:
df_2013.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 1 to 76
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ice class            74 non-null     object
 1   grt (tons)           73 non-null     object
 2   vessel operator      74 non-null     object
 3   cargo                76 non-null     object
 4   port of destination  75 non-null     object
 5   time elapsed on nsr  73 non-null     object
 6   avg speed (knots)    73 non-null     object
 7   vessel               75 non-null     object
 8   flag                 66 non-null     object
 9   starting port        75 non-null     object
 10  starting date        66 non-null     object
 11  starting time        53 non-null     object
 12  nsr entry date       75 non-null     object
 13  nsr entry time       73 non-null     object
 14  nsr entry location   72 non-null     object
 15  nsr exit date        75 non-null     object
 16  nsr exit t

In [52]:
# remove superfluous header rows

df_2013.drop(11,axis=0,inplace=True)
df_2013.drop(23,axis=0,inplace=True)
df_2013.drop(36,axis=0,inplace=True)
df_2013.drop(37,axis=0,inplace=True)
df_2013.drop(49,axis=0,inplace=True)
df_2013.drop(63,axis=0,inplace=True)


In [53]:
# correct cells that do not follow format

df_2013.loc[1,'port of destination'] = 'Pevek-Nakhodka'

df_2013.loc[7,'starting port'] = 'Beilun, China'
df_2013.loc[7,'starting date'] = '10.07.13'
df_2013.loc[7,'starting time'] = '02:00'

df_2013.loc[9,'starting port'] = 'Petropavlovsk-Kamchatsky'
df_2013.loc[9,'starting date'] = np.nan
df_2013.loc[9,'starting time'] = np.nan

df_2013.loc[10,'vessel'] = 'mv "Georgiy Ushakov"'
df_2013.loc[10,'flag'] = 'Russia'

df_2013.loc[12,'starting port'] = 'Mongstad, Norway'
df_2013.loc[12, 'starting date'] = '17.07.13'
df_2013.loc[12,'starting time'] = np.nan

df_2013.loc[15,'starting port'] = 'Ob Bay'
df_2013.loc[15,'starting time'] = '22:00'
df_2013.loc[15,'starting date'] = '26.07.13'

df_2013.loc[22,'starting port'] = 'Tan Cang Cai Mep, Vietnam'
df_2013.loc[22,'starting date'] = np.nan
df_2013.loc[22,'starting time'] = np.nan

df_2013.loc[26,'starting port'] = 'Ob Bay'
df_2013.loc[26,'starting time'] = '12:00'
df_2013.loc[26,'starting date'] = '13.08.13'

df_2013.loc[29,'starting port'] = 'Ob Bay'
df_2013.loc[29,'starting time'] = '20:00'
df_2013.loc[29,'starting date'] = '25.08.13'

df_2013.loc[35,'starting port'] = 'Petropavlovsk-Kamchatsky'
df_2013.loc[35,'starting time'] = np.nan
df_2013.loc[35,'starting date'] = '03.09.13'

df_2013.loc[41,'starting port'] = 'Alesund, Norway'
df_2013.loc[41,'starting time'] = np.nan
df_2013.loc[41,'starting date'] = np.nan

df_2013.loc[43,'starting port'] = 'c. Shmidt'
df_2013.loc[43,'starting time'] = '20:00'
df_2013.loc[43,'starting date'] = '16.09.13'

df_2013.loc[45,'starting port'] = 'Ob Bay'
df_2013.loc[45,'starting time'] = '12:00'
df_2013.loc[45,'starting date'] = '18.08.13'

df_2013.loc[46,'starting port'] = 'Hammerfest, Norway'
df_2013.loc[46,'starting time'] = np.nan
df_2013.loc[46,'starting date'] = '13.09.13'

df_2013.loc[48,'starting port'] = 'Petropavlovsk-Kamchatsky'
df_2013.loc[48,'starting time'] = '12:50'
df_2013.loc[48,'starting date'] = '10.09.13'
df_2013.loc[48,'vessel'] = 'mv "Inzhener Trubin"'
df_2013.loc[48,'port of destination'] = 'Arkhangelsk'
df_2013.loc[48,'cargo'] = 'general cargo'
df_2013.loc[48,'flag'] = 'Russia'
df_2013.loc[48,'nsr entry date'] = '23.09.13'
df_2013.loc[48,'nsr exit date'] = '28.10.13'
df_2013.loc[48,'nsr entry time'] = '18:00'
df_2013.loc[48,'nsr exit time'] = '12:45'
df_2013.loc[48,'nsr entry location'] = 'c. Dezhnev'
df_2013.loc[48,'nsr exit location'] = 'Kara Gate'
df_2013.drop(50,axis=0,inplace=True)

In [54]:
df_2013

,ice class,grt (tons),vessel operator,cargo,port of destination,time elapsed on nsr,avg speed (knots),vessel,flag,starting port,starting date,starting time,nsr entry date,nsr entry time,nsr entry location,nsr exit date,nsr exit time,nsr exit location,cargo mass
1,Arc5,11 290,MSCO / Chukotsnab,diesel fuel,Pevek-Nakhodka,15.3,7.0,"mt ""Varzuga""",RUSSIA,Murmansk,25.06.13,00:00,28.06.13,16:00,c. Zhelaniya,14.07.13,14:00,c. Dezhnev,13 658
2,Arc5,11 290,MSCO / Chukotsnab,diesel fuel,Pevek,11.7,9.2,"mt ""Indiga""",RUSSIA,Murmansk,25.06.13,19:00,28.06.13,17:00,c. Zhelaniya,10.07.13,09:20,Pevek,13 645
3,Ice-1A (Arc4),40 142,Nordic Bulk Carriers,iron ore Eurochem,Lanshan. China,19.9,5.4,"mv ""Nordic Orion""",PANAMA,Murmansk,01.07.13,01:30,03.07.13,10:00,c. Zhelaniya,23.07.13,08:55,c. Dezhnev,66 000
4,Arc4,5 025,Khatanga Sea Merchant Port CJSC,diesel fuel,Khatanga Bay,25.2,4.3,"mv ""Boris Vilkitskiy""",RUSSIA,NaN,NaN,NaN,13.07.13,12:00,Yugorski Shar,07.08.13,24:00,Khatanga Bay,5 996
5,Arc5,11 290,MSCO/ Chukotsnab,ballast,Ob Bay,11.3,7.2,"mt ""Indiga""",RUSSIA,Pevek,15.07.13,01:30,15.07.13,01:30,Pevek,24.07.13,12:50,Ob Bay,FD 22 654 tones
6,Arc4,4 110,Sky LLC,diesel fuel,Ob Bay,20.6,7.1,"mt ""Egvekinot""",RUSSIA,Provideniya,None,None,17.07.13,23:00,c. Dezhnev,07.08.13,12:00,None,4 646
7,Ice 1A (Arc4),40 142,Nordic Bulk Carriers,ballast,Murmansk,16.4,8.0,"mv ""Nordic Odyssey""",PANAMA,"Beilun, China",10.07.13,02:00,18.07.13,01:00,c. Dezhnev,03.08.13,10:30,Kara Gate,FD 87 216 tones
8,Arc5,6 418,Northern Shipping Company,general cargo,Petropavlovsk- Kamchatsky,14.1,8.0,"mv ""Inzhener Trubin""",RUSSIA,Arkhangelsk,19.07.13,20:30,21.07.13,20:00,Kara Gate,14.08.13,04:10,c. Dezhnev,2 240
9,Arc5,13 204,Vostokbunker CJSC,diesel fuel,Murmansk,18.8,8.8,"mt ""Bukhta Slavyanka""",RUSSIA,Petropavlovsk-Kamchatsky,NaN,NaN,23.07.13,10:00,c. Dezhnev,11.08.13,06:00,Kara Gate,7 923
10,Arc4,6 204,Khatanga Sea Merchant Port CJSC,general cargo,Anabar Bay,15.0,6.4,"mv ""Georgiy Ushakov""",Russia,Arkhangelsk,None,None,23.07.13,23:00,Kara Gate,07.08.13,23:00,Anabar Bay,6 697


In [55]:
# continue on manual fixes

# df_2013.loc[51,'starting port'] = 'Ulsan, South Korea'
# df_2013.loc[51,'starting date'] = '14.09.13'
# df_2013.loc[51,'starting time'] = np.nan

# df_2013.loc[52,'starting port'] = 'Ob Bay'
# df_2013.loc[52,'starting date'] = '26.09.13'
# df_2013.loc[52,'starting time'] = '18:00'

# df_2013.loc[53,'starting port'] = 'Mongstad, Norway'
# df_2013.loc[53,'starting date'] = '21.09.13'
# df_2013.loc[53,'starting time'] = np.nan

# df_2013.loc[55,'starting port'] = 'Kirkenes, Norway'
# df_2013.loc[55,'starting date'] = np.nan
# df_2013.loc[55,'starting time'] = np.nan

# df_2013.loc[57,'starting port'] = 'Onsan, South Korea'
# df_2013.loc[57,'starting date'] = np.nan
# df_2013.loc[57,'starting time'] = np.nan

# df_2013.loc[64,'vessel'] = 'mt "Viktor Bakaev"'
# df_2013.loc[64,'port of destination'] = 'Rotterdam, Netherlands'
# df_2013.loc[64,'cargo'] = 'ballast'
# df_2013.loc[64,'flag'] = 'Liberia'
# df_2013.loc[64,'nsr entry date'] = '23.09.13'
# df_2013.loc[64,'nsr exit date'] = '28.10.13'
# df_2013.loc[64,'nsr entry time'] = '18:00'
# df_2013.loc[64,'nsr exit time'] = '12:45'
# df_2013.loc[64,'nsr entry location'] = 'c. Dezhnev'
# df_2013.loc[64,'nsr exit location'] = 'Kara Gate'
# df_2013.loc[64,'starting port'] = 'Yosu, South Korea'
# df_2013.loc[64,'starting date'] = '2.10.13'
# df_2013.loc[64,'starting time'] = '09:00'
# drop 62?

In [56]:
# change column types for non-string columns

# df_2013['grt'] = df_2013['grt (tons)'].str.replace(' ','',regex=True)
# df_2013['grt'] = pd.to_numeric(df_2013['grt'])
# df_2013.drop('grt (tons)',axis=1,inplace=True)

# df_2013['time elapsed on nsr'] = pd.to_numeric(df_2013['time elapsed on nsr'])
# df_2013['avg speed'] = pd.to_numeric(df_2013['avg speed (knots)'])
# df_2013.drop('avg speed (knots)',axis=1,inplace=True)

# # pull out digits from string with regex
# ser_exp = [df_2013['starting date'],df_2013['nsr entry date'],df_2013['nsr exit date']]

# for c in ser_exp:
#     try:
#         c = pd.to_datetime

# df_2013['cargo mass'] = df_2013['cargo mass'].str.extract(r'((?:\d+\s*)+)')
# df_2013['cargo mass'] = df_2013['cargo mass'].str.replace(' ','')
# df_2013['cargo mass'] = pd.to_numeric(df_2013['cargo mass'])

# df_2013['starting date'] = pd.to_datetime(df_2013['starting date'],format='%d.%m.%y')
# df_2013['nsr entry date'] = pd.to_datetime(df_2013['nsr entry date'],format='%d.%m.%y')
# df_2013['nsr exit date'] = pd.to_datetime(df_2013['nsr exit date'],format='%d.%m.%y')

# df_2013['starting time'] = pd.to_datetime(df_2013['starting time'],format='%H:%M')
# df_2013['nsr entry time'] = pd.to_datetime(df_2013['nsr entry time'],format='%H:%M')
# df_2013['nsr exit time'] = pd.to_datetime(df_2013['nsr exit time'],format='%H:%M')

# df_2013['starting time'] = df_2013['starting time'].dt.strftime('%H:%M')
# df_2013['nsr entry time'] = df_2013['nsr entry time'].dt.strftime('%H:%M')
# df_2013['nsr exit time'] = df_2013['nsr exit time'].dt.strftime('%H:%M')

# df_2013.head()
df_2013.columns = ['ice class','grt','ship operator','cargo type','port of arrival',
                   'time elapsed on nsr','avg speed (knots)','vessel name','flag','port of departure',
                   'date of departure','time of departure','nsr entry date','nsr entry time',
                   'nsr entry location','nsr exit date','nsr exit time','nsr exit location',
                   'approx cargo weight (tn)'
            ]

temp_df_2013 = df_2013.copy()



In [57]:
df_2013.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69 entries, 1 to 76
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   ice class                 69 non-null     object
 1   grt                       68 non-null     object
 2   ship operator             68 non-null     object
 3   cargo type                69 non-null     object
 4   port of arrival           69 non-null     object
 5   time elapsed on nsr       68 non-null     object
 6   avg speed (knots)         68 non-null     object
 7   vessel name               69 non-null     object
 8   flag                      67 non-null     object
 9   port of departure         68 non-null     object
 10  date of departure         58 non-null     object
 11  time of departure         44 non-null     object
 12  nsr entry date            69 non-null     object
 13  nsr entry time            68 non-null     object
 14  nsr entry location        67 non-

In [58]:
# 2014

df_2014 = data[2014]
df_2014.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9
0,0,Name of vessel,NaN,Flag,Type,GRT,Date and place of entering the\nNSR water area,NaN,Date and place of leaving the\nNSR water area,NaN,Days spent\nat NSR
1,1,List of the vessels crossed both Western and E...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1,Kapitan Khlebnikov,Russia,Passengers Ship,12 288,28.06.14,c. Dezhnev,c. Zhelania,28.07.14,30
3,3,2,Tor Viking,Sweden,Tug / Supply / Icebreaker,3 382,03.07.14,c. Zhelania,c. Dezhnev,16.07.14,13
4,4,3,Anichkov Bridge,Russia,Oil/Chemical Tanker,27 829,05.07.14,c. Dezhnev,c. Zhelania,17.08.14,43


In [59]:
df_2014.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9
51,51,18,Varzuga,Russia,Crude Oil Tanker,11 290,17.10.14,Karskiye Vorota,port Pevek,28.10.14,11
52,52,19,Indiga,Russia,Crude Oil Tanker,11 290,20.10.14,port Pevek,c. Zhelania,17.11.14,28
53,53,20,Varzuga,Russia,Crude Oil Tanker,11 290,31.10.14,port Pevek,c. Zhelania,17.11.14,17
54,54,21,Indiga,Russia,Crude Oil Tanker,11 290,04.11.14,port Pevek,c. Zhelania,17.11.14,13
55,55,22,Ivan Ryabov,Russia,General Cargo,5 370,06.11.14,port Pevek,c. Zhelania,16.11.14,10


In [60]:
# rename columns to get closer to standardization

df_2014.columns = ['index','index2','vessel name','flag','cargo type','grt','nsr entry date','nsr entry location','nsr exit location','nsr exit date','time elapsed on nsr']
df_2014.head()

,index,index2,vessel name,flag,cargo type,grt,nsr entry date,nsr entry location,nsr exit location,nsr exit date,time elapsed on nsr
0,0,Name of vessel,NaN,Flag,Type,GRT,Date and place of entering the\nNSR water area,NaN,Date and place of leaving the\nNSR water area,NaN,Days spent\nat NSR
1,1,List of the vessels crossed both Western and E...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1,Kapitan Khlebnikov,Russia,Passengers Ship,12 288,28.06.14,c. Dezhnev,c. Zhelania,28.07.14,30
3,3,2,Tor Viking,Sweden,Tug / Supply / Icebreaker,3 382,03.07.14,c. Zhelania,c. Dezhnev,16.07.14,13
4,4,3,Anichkov Bridge,Russia,Oil/Chemical Tanker,27 829,05.07.14,c. Dezhnev,c. Zhelania,17.08.14,43


In [61]:
# drop irrelevant columns and rows
df_2014.drop('index',axis=1,inplace=True)
df_2014.drop('index2',axis=1,inplace=True)
df_2014.drop(0,axis=0,inplace=True)
df_2014.drop(1,axis=0,inplace=True)

df_2014.reset_index(drop=True,inplace=True)

df_2014.head()

,vessel name,flag,cargo type,grt,nsr entry date,nsr entry location,nsr exit location,nsr exit date,time elapsed on nsr
0,Kapitan Khlebnikov,Russia,Passengers Ship,12 288,28.06.14,c. Dezhnev,c. Zhelania,28.07.14,30
1,Tor Viking,Sweden,Tug / Supply / Icebreaker,3 382,03.07.14,c. Zhelania,c. Dezhnev,16.07.14,13
2,Anichkov Bridge,Russia,Oil/Chemical Tanker,27 829,05.07.14,c. Dezhnev,c. Zhelania,17.08.14,43
3,Oden,Sweden,Icebreaker/Research,9 605,09.07.14,North of c. Zhelania,North of c. Dezhnev,17.08.14,39
4,Yaroslav Mydryy,Russia,Oil/Chemical Tanker,6 262,09.07.14,c. Dezhnev,c. Zhelania,28.07.14,19


In [62]:
df_2014 = basic_clean(df_2014)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name            1
flag                   1
cargo type             1
grt                    1
nsr entry date         1
nsr entry location     1
nsr exit location      1
nsr exit date          1
time elapsed on nsr    1
dtype: int64


In [63]:
# 2015

df_2015 = data[2015]
df_2015.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0,№,Vessel name,Flag,Shipowner,Ice class,Type,Cargo,Qty (mt) /\npassangers,Max\ndraught\n(m),Depature,Destination,I/b\nassistance,WBNSR\n(Kara Gate),WBNSR\n(Cape Zhelaniya),EBNSR\n(Cape Dezhnev),NSR\npassage\ntime (days)
1,1,1,Yasnyi (with\nfloating crane СПК-\n42150),Russia,"FBI ""MRS""",Arc 5\n(crane Ice2),Rescue,NaN,0,"4,9",Saint-\nPetersbourg,Petropavlovsk-\nKamchatskiy,No,30.07.15 2:18,NaN,15.08.15 23:30,"16,9"
2,2,2,Yong Sheng,China,Cosco Shipping,Arc 4,General\ncargo\ncarrier,Steel coils; Project\nand Windmill tower\nand ...,12 716,"8,0",Shanghai (China),Varberg\n(Sweden),Taymyr\nAtomflot,NaN,10.08.15 15:20,02.08.15 3:30,"8,5"
3,3,3,Winter Bay,St. Kitts &\nNevis,Dalriada Ltd,Ice 1,Reefer,Frozen fish & meat,1 938,"6,2",Tromse\n(Norway),Osaka\n(Japan),No,NaN,06.08.15 15:00,16.08.15 8:23,"9,7"
4,4,4,Garmonia,Russia,Yuzmorrybflot\nLtd.,Arc 4,Reefer,Frozen fish,2 806,"7,2",Nakhodka,Murmansk,Yamal\nAtomflot,24.08.15 19:45,NaN,14.08.15 12:23,"10,3"


In [64]:
df_2015.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  20 non-null     int64 
 1   0           20 non-null     object
 2   1           19 non-null     object
 3   2           19 non-null     object
 4   3           19 non-null     object
 5   4           19 non-null     object
 6   5           19 non-null     object
 7   6           13 non-null     object
 8   7           20 non-null     object
 9   8           20 non-null     object
 10  9           19 non-null     object
 11  10          19 non-null     object
 12  11          19 non-null     object
 13  12          13 non-null     object
 14  13          7 non-null      object
 15  14          19 non-null     object
 16  15          20 non-null     object
dtypes: int64(1), object(16)
memory usage: 2.8+ KB


In [65]:
df_2015.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
15,15,15,Volk Arktiki,Russia,LLC\nRiboloveckaya\nArtel Vilychinsk,Ice 3,Fishinng,Ballast,0,"4,0",Petropavlovsk-\nKamchatskiy,Murmansk,No,28.10.15 10:04,NaN,14.10.15 2:30,"14,3"
16,16,16,Palladiy,Russia,Valkur Ltd,Arc 5,Tanker,Ballast,0,"6,4",Petropavlovsk-\nKamchatskiy,Murmansk,Vaigach\nAtomflot,10.11.15 9:28,NaN,25.10.15 2:00,"16,3"
17,17,17,Kigoriak,Russia,Femco Mng.,Ib 6,Iceb. Tug,NaN,0,"8,3",Shanghai (China),Murmansk,No,22.11.15 2:50,NaN,08.11.15 6:20,"13,9"
18,18,18,Tor Viking,Sweden,Viking Supply\nShips A/S,Arc 7,Iceb. Tug,NaN,0,"7,2",Seattle\n(USA),L andskrona\n(Sweden),No,NaN,10.12.15 12:00,29.11.15 7:10,"11,2"
19,19,Total,NaN,NaN,NaN,NaN,NaN,NaN,39 586/137,Average,NaN,NaN,NaN,NaN,NaN,NaN,"10,6"


In [66]:
# rename columns to closer adhere to standardization

df_2015.columns = ['index','index2','vessel name','flag','ship operator',
                   'ice class','ship type','cargo type','approx cargo weight (tn)',
                   'max draught','port of departure','port of arrival',
                   'icebreaker assistance','Kara Gate date and time','Cape Zhelaniya date and time',
                   'Cape Dezhnev date and time','time elapsed on nsr'
                  ]
df_2015.head()

,index,index2,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),max draught,port of departure,port of arrival,icebreaker assistance,Kara Gate date and time,Cape Zhelaniya date and time,Cape Dezhnev date and time,time elapsed on nsr
0,0,№,Vessel name,Flag,Shipowner,Ice class,Type,Cargo,Qty (mt) /\npassangers,Max\ndraught\n(m),Depature,Destination,I/b\nassistance,WBNSR\n(Kara Gate),WBNSR\n(Cape Zhelaniya),EBNSR\n(Cape Dezhnev),NSR\npassage\ntime (days)
1,1,1,Yasnyi (with\nfloating crane СПК-\n42150),Russia,"FBI ""MRS""",Arc 5\n(crane Ice2),Rescue,NaN,0,"4,9",Saint-\nPetersbourg,Petropavlovsk-\nKamchatskiy,No,30.07.15 2:18,NaN,15.08.15 23:30,"16,9"
2,2,2,Yong Sheng,China,Cosco Shipping,Arc 4,General\ncargo\ncarrier,Steel coils; Project\nand Windmill tower\nand ...,12 716,"8,0",Shanghai (China),Varberg\n(Sweden),Taymyr\nAtomflot,NaN,10.08.15 15:20,02.08.15 3:30,"8,5"
3,3,3,Winter Bay,St. Kitts &\nNevis,Dalriada Ltd,Ice 1,Reefer,Frozen fish & meat,1 938,"6,2",Tromse\n(Norway),Osaka\n(Japan),No,NaN,06.08.15 15:00,16.08.15 8:23,"9,7"
4,4,4,Garmonia,Russia,Yuzmorrybflot\nLtd.,Arc 4,Reefer,Frozen fish,2 806,"7,2",Nakhodka,Murmansk,Yamal\nAtomflot,24.08.15 19:45,NaN,14.08.15 12:23,"10,3"


In [67]:
# drop irrelevant columns and rows

df_2015.drop(19,axis=0,inplace=True)
df_2015.drop(0,axis=0,inplace=True)
df_2015.drop('index',axis=1,inplace=True)
df_2015.drop('index2',axis=1,inplace=True)

df_2015.head()

,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),max draught,port of departure,port of arrival,icebreaker assistance,Kara Gate date and time,Cape Zhelaniya date and time,Cape Dezhnev date and time,time elapsed on nsr
1,Yasnyi (with\nfloating crane СПК-\n42150),Russia,"FBI ""MRS""",Arc 5\n(crane Ice2),Rescue,NaN,0,"4,9",Saint-\nPetersbourg,Petropavlovsk-\nKamchatskiy,No,30.07.15 2:18,NaN,15.08.15 23:30,"16,9"
2,Yong Sheng,China,Cosco Shipping,Arc 4,General\ncargo\ncarrier,Steel coils; Project\nand Windmill tower\nand ...,12 716,"8,0",Shanghai (China),Varberg\n(Sweden),Taymyr\nAtomflot,NaN,10.08.15 15:20,02.08.15 3:30,"8,5"
3,Winter Bay,St. Kitts &\nNevis,Dalriada Ltd,Ice 1,Reefer,Frozen fish & meat,1 938,"6,2",Tromse\n(Norway),Osaka\n(Japan),No,NaN,06.08.15 15:00,16.08.15 8:23,"9,7"
4,Garmonia,Russia,Yuzmorrybflot\nLtd.,Arc 4,Reefer,Frozen fish,2 806,"7,2",Nakhodka,Murmansk,Yamal\nAtomflot,24.08.15 19:45,NaN,14.08.15 12:23,"10,3"
5,Happy Sky,Netherlands,BigLift Shipping\nB.V.,Arc 4,General\ncargo\ncarrier,Ballast,0,"6,9",Kirkines\n(Norway),Masan\n(South Korea),No,27.08.15 14:45,NaN,04.09.15 7:25,"7,7"


In [68]:
df_2015 = basic_clean(df_2015)



[ vessel name 1 ]
[ vessel name 8 ]
New line characters have been removed. Detected 2 cell(s) with non-ASCII characters.
Null Count:
 vessel name                      0
flag                             0
ship operator                    0
ice class                        0
ship type                        0
cargo type                       6
approx cargo weight (tn)         0
max draught                      0
port of departure                0
port of arrival                  0
icebreaker assistance            0
Kara Gate date and time          6
Cape Zhelaniya date and time    12
Cape Dezhnev date and time       0
time elapsed on nsr              0
dtype: int64


In [69]:
df_2015.head()

,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),max draught,port of departure,port of arrival,icebreaker assistance,Kara Gate date and time,Cape Zhelaniya date and time,Cape Dezhnev date and time,time elapsed on nsr
1,Yasnyi (with floating crane СПК- 42150),Russia,"FBI ""MRS""",Arc 5 (crane Ice2),Rescue,NaN,0,4.9,Saint- Petersbourg,Petropavlovsk- Kamchatskiy,No,30.07.15 2:18,NaN,15.08.15 23:30,16.9
2,Yong Sheng,China,Cosco Shipping,Arc 4,General cargo carrier,Steel coils; Project and Windmill tower and bl...,12 716,8.0,Shanghai (China),Varberg (Sweden),Taymyr Atomflot,NaN,10.08.15 15:20,02.08.15 3:30,8.5
3,Winter Bay,St. Kitts & Nevis,Dalriada Ltd,Ice 1,Reefer,Frozen fish & meat,1 938,6.2,Tromse (Norway),Osaka (Japan),No,NaN,06.08.15 15:00,16.08.15 8:23,9.7
4,Garmonia,Russia,Yuzmorrybflot Ltd.,Arc 4,Reefer,Frozen fish,2 806,7.2,Nakhodka,Murmansk,Yamal Atomflot,24.08.15 19:45,NaN,14.08.15 12:23,10.3
5,Happy Sky,Netherlands,BigLift Shipping B.V.,Arc 4,General cargo carrier,Ballast,0,6.9,Kirkines (Norway),Masan (South Korea),No,27.08.15 14:45,NaN,04.09.15 7:25,7.7


In [70]:
# expand checkpoint fields separating date and time

df_2015[['Kara Gate date','Kara Gate time']] = df_2015['Kara Gate date and time'].str.split(n=1,expand=True)
df_2015.drop('Kara Gate date and time',axis=1,inplace=True)

df_2015[['Cape Zhelaniya date','Cape Zhelaniya time']] = df_2015['Cape Zhelaniya date and time'].str.split(n=1,expand=True)
df_2015.drop('Cape Zhelaniya date and time',axis=1,inplace=True)

df_2015[['Cape Dezhnev date','Cape Deshnev time']] = df_2015['Cape Dezhnev date and time'].str.split(n=1,expand=True)
df_2015.drop('Cape Dezhnev date and time',axis=1,inplace=True)

In [71]:
df_2015.head()

,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),max draught,port of departure,port of arrival,icebreaker assistance,time elapsed on nsr,Kara Gate date,Kara Gate time,Cape Zhelaniya date,Cape Zhelaniya time,Cape Dezhnev date,Cape Deshnev time
1,Yasnyi (with floating crane СПК- 42150),Russia,"FBI ""MRS""",Arc 5 (crane Ice2),Rescue,NaN,0,4.9,Saint- Petersbourg,Petropavlovsk- Kamchatskiy,No,16.9,30.07.15,2:18,NaN,NaN,15.08.15,23:30
2,Yong Sheng,China,Cosco Shipping,Arc 4,General cargo carrier,Steel coils; Project and Windmill tower and bl...,12 716,8.0,Shanghai (China),Varberg (Sweden),Taymyr Atomflot,8.5,NaN,NaN,10.08.15,15:20,02.08.15,3:30
3,Winter Bay,St. Kitts & Nevis,Dalriada Ltd,Ice 1,Reefer,Frozen fish & meat,1 938,6.2,Tromse (Norway),Osaka (Japan),No,9.7,NaN,NaN,06.08.15,15:00,16.08.15,8:23
4,Garmonia,Russia,Yuzmorrybflot Ltd.,Arc 4,Reefer,Frozen fish,2 806,7.2,Nakhodka,Murmansk,Yamal Atomflot,10.3,24.08.15,19:45,NaN,NaN,14.08.15,12:23
5,Happy Sky,Netherlands,BigLift Shipping B.V.,Arc 4,General cargo carrier,Ballast,0,6.9,Kirkines (Norway),Masan (South Korea),No,7.7,27.08.15,14:45,NaN,NaN,04.09.15,7:25


In [72]:
df_2015.tail()

,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),max draught,port of departure,port of arrival,icebreaker assistance,time elapsed on nsr,Kara Gate date,Kara Gate time,Cape Zhelaniya date,Cape Zhelaniya time,Cape Dezhnev date,Cape Deshnev time
14,Yasnyi,Russia,"FBI ""MRS""",Arc 5,Rescue,NaN,0,4.9,Petropavlovsk- Kamchatskiy,Murmansk,No,12.7,23.10.15,17:11,NaN,NaN,11.10.15,0:05
15,Volk Arktiki,Russia,LLC Riboloveckaya Artel Vilychinsk,Ice 3,Fishinng,Ballast,0,4.0,Petropavlovsk- Kamchatskiy,Murmansk,No,14.3,28.10.15,10:04,NaN,NaN,14.10.15,2:30
16,Palladiy,Russia,Valkur Ltd,Arc 5,Tanker,Ballast,0,6.4,Petropavlovsk- Kamchatskiy,Murmansk,Vaigach Atomflot,16.3,10.11.15,9:28,NaN,NaN,25.10.15,2:00
17,Kigoriak,Russia,Femco Mng.,Ib 6,Iceb. Tug,NaN,0,8.3,Shanghai (China),Murmansk,No,13.9,22.11.15,2:50,NaN,NaN,08.11.15,6:20
18,Tor Viking,Sweden,Viking Supply Ships A/S,Arc 7,Iceb. Tug,NaN,0,7.2,Seattle (USA),L andskrona (Sweden),No,11.2,NaN,NaN,10.12.15,12:00,29.11.15,7:10


In [73]:
# 2016

df_2016 = data[2016]

df_2016.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,0,NaN,№,Vessel name,Flag,Shipowner,Ice class,Type,Cargo,Qty (mt) /\npassangers,DWT\n(tons),Max\ndraught\n(m),Port of\ndeparture,Port of\ndestination,I/b\nassistance,WBNSR\n(Kara Gate),WBNSR\n(Yugorskiy\nShar),WBNSR (Cape\nZhelaniya),EBNSR\n(Cape Dezhnev),NSR\npassage\ntime\n(days)
1,1,NaN,1,Kapitan Khlebnikov,Russia,"PJSC ""Far Eastern\nShipping Company""",Icebreaker 7,Icebreaker,Passangers,120,4 418,"8,5",Anadyr,Murmansk,No,NaN,NaN,27.07.16 21:35,16.07.16 2:30,"11,8"
2,2,NaN,2,Brage Viking,Russia,"LLC ""Viking Supply""",Arc 5,Tug,NaN,0,4 352,"7,4",Kholmsk,Aberdeen\n(Scotland),Vaigach\n(Atomflot),NaN,NaN,06.08.16 7:00,21.07.16 18:30,"15,5"
3,3,NaN,3,Yong Sheng,Hong Kong,Cosco Shipping Ltd,Arc 4,General cargo\ncarrier,"Ore in bulk,steel",13 514,19 150,"8,3",Shanghai\n(China),Glasgow\n(Scotland),Vaigach\n(Atomflot),NaN,NaN,06.08.16 13:26,28.07.16 7:00,"9,3"
4,4,NaN,4,Spasatel Karev,Russia,FBI “Marine Rescue\nService of\nRosmorrechflot”,Arc 5,Rescue,NaN,0,1 109,NaN,Saint-\nPetersbourg,Vladivostok,No,NaN,30.07.16 6:30,NaN,20.08.16 9:50,"21,1"


In [74]:
df_2016.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
16,16,NaN,17,Normann,Liberia,LLC ''Femko\nmanagement'',Arc 7,Supply vessel,NaN,0,2 600,"7,6",Bergen\n(Norway),Kholmsk,NaN,NaN,NaN,02.11.16 12:00,17.11.16 17:45,"15,2"
17,17,NaN,18,HHL Valparaiso,Liberia,Hansa Heavy Lift,Arc 4,General cargo\ncarrier,Port crandes; sand in\nbig-bags; invertors,4 097,19 413,"8,8",Saint-\nPetersbourg,Vostochniy,50 let Pobedy\n(Atomflot),NaN,NaN,03.11.16 20:30,11.11.16 19:06,"7,9"
18,18,NaN,19,Pomor,Liberia,LLC ''Femko\nmanagement'',Arc 7,Supply vessel,NaN,0,2 600,"6,1",Bergen\n(Norway),Kholmsk,NaN,NaN,NaN,08.11.16 8:10,17.11.16 18:20,"9,4"
19,19,NaN,Total,NaN,Cargo,NaN,NaN,NaN,NaN,214 513,Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"14,3"
20,20,NaN,NaN,NaN,Passangers,NaN,NaN,NaN,NaN,246,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
# standardize column names

df_2016.columns = ['index','index2','index3','vessel name','flag','ship operator','ice class',
                   'ship type','cargo type','approx cargo weight (tn)','dead weight tonnage',
                   'max draught','port of departure','port of arrival','icebreaker assistance',
                   'Kara Gate date and time','Yugorskiy Strait date and time','Cape Zhelaniya date and time',
                   'Cape Dezhnev date and time','time elapsed on nsr']

df_2016.head()

,index,index2,index3,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),dead weight tonnage,max draught,port of departure,port of arrival,icebreaker assistance,Kara Gate date and time,Yugorskiy Strait date and time,Cape Zhelaniya date and time,Cape Dezhnev date and time,time elapsed on nsr
0,0,NaN,№,Vessel name,Flag,Shipowner,Ice class,Type,Cargo,Qty (mt) /\npassangers,DWT\n(tons),Max\ndraught\n(m),Port of\ndeparture,Port of\ndestination,I/b\nassistance,WBNSR\n(Kara Gate),WBNSR\n(Yugorskiy\nShar),WBNSR (Cape\nZhelaniya),EBNSR\n(Cape Dezhnev),NSR\npassage\ntime\n(days)
1,1,NaN,1,Kapitan Khlebnikov,Russia,"PJSC ""Far Eastern\nShipping Company""",Icebreaker 7,Icebreaker,Passangers,120,4 418,"8,5",Anadyr,Murmansk,No,NaN,NaN,27.07.16 21:35,16.07.16 2:30,"11,8"
2,2,NaN,2,Brage Viking,Russia,"LLC ""Viking Supply""",Arc 5,Tug,NaN,0,4 352,"7,4",Kholmsk,Aberdeen\n(Scotland),Vaigach\n(Atomflot),NaN,NaN,06.08.16 7:00,21.07.16 18:30,"15,5"
3,3,NaN,3,Yong Sheng,Hong Kong,Cosco Shipping Ltd,Arc 4,General cargo\ncarrier,"Ore in bulk,steel",13 514,19 150,"8,3",Shanghai\n(China),Glasgow\n(Scotland),Vaigach\n(Atomflot),NaN,NaN,06.08.16 13:26,28.07.16 7:00,"9,3"
4,4,NaN,4,Spasatel Karev,Russia,FBI “Marine Rescue\nService of\nRosmorrechflot”,Arc 5,Rescue,NaN,0,1 109,NaN,Saint-\nPetersbourg,Vladivostok,No,NaN,30.07.16 6:30,NaN,20.08.16 9:50,"21,1"


In [76]:
# drop unnecessary columns

df_2016.drop(['index','index2','index3'],axis=1,inplace=True)
df_2016.drop([0,19,20],axis=0,inplace=True)

df_2016.tail()


,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),dead weight tonnage,max draught,port of departure,port of arrival,icebreaker assistance,Kara Gate date and time,Yugorskiy Strait date and time,Cape Zhelaniya date and time,Cape Dezhnev date and time,time elapsed on nsr
14,Winter Bay,St. Kitts &\nNevis,Darliada Ltd,Ice 1,Refeer,Frozen fish,1 874,2 050,"5,8",Petropavlovsk-\nKamchatskiy,Saint-\nPetersbourg,No,NaN,NaN,11.10.16 11:45,03.10.16 1:25,"8,4"
15,Inzhener Trubin,Russia,Northern Shipping\nCompany,Arc 5,General cargo\ncarrier,Ballast,0,7 075,"5,8",Arhangelsk,Quingdao\n(China),50 let Pobedy\n(Atomflot),31.10.16 16:50,NaN,NaN,12.11.16 8:55,"11,7"
16,Normann,Liberia,LLC ''Femko\nmanagement'',Arc 7,Supply vessel,NaN,0,2 600,"7,6",Bergen\n(Norway),Kholmsk,NaN,NaN,NaN,02.11.16 12:00,17.11.16 17:45,"15,2"
17,HHL Valparaiso,Liberia,Hansa Heavy Lift,Arc 4,General cargo\ncarrier,Port crandes; sand in\nbig-bags; invertors,4 097,19 413,"8,8",Saint-\nPetersbourg,Vostochniy,50 let Pobedy\n(Atomflot),NaN,NaN,03.11.16 20:30,11.11.16 19:06,"7,9"
18,Pomor,Liberia,LLC ''Femko\nmanagement'',Arc 7,Supply vessel,NaN,0,2 600,"6,1",Bergen\n(Norway),Kholmsk,NaN,NaN,NaN,08.11.16 8:10,17.11.16 18:20,"9,4"


In [77]:
df_2016 = basic_clean(df_2016)

[ ship operator 4 ]
New line characters have been removed. Detected 1 cell(s) with non-ASCII characters.
Null Count:
 vessel name                        0
flag                               0
ship operator                      0
ice class                          0
ship type                          0
cargo type                         6
approx cargo weight (tn)           0
dead weight tonnage                0
max draught                        1
port of departure                  0
port of arrival                    0
icebreaker assistance              3
Kara Gate date and time           15
Yugorskiy Strait date and time    17
Cape Zhelaniya date and time       4
Cape Dezhnev date and time         0
time elapsed on nsr                0
dtype: int64


In [78]:
df_2016.head()

,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),dead weight tonnage,max draught,port of departure,port of arrival,icebreaker assistance,Kara Gate date and time,Yugorskiy Strait date and time,Cape Zhelaniya date and time,Cape Dezhnev date and time,time elapsed on nsr
1,Kapitan Khlebnikov,Russia,"PJSC ""Far Eastern Shipping Company""",Icebreaker 7,Icebreaker,Passangers,120,4 418,8.5,Anadyr,Murmansk,No,NaN,NaN,27.07.16 21:35,16.07.16 2:30,11.8
2,Brage Viking,Russia,"LLC ""Viking Supply""",Arc 5,Tug,NaN,0,4 352,7.4,Kholmsk,Aberdeen (Scotland),Vaigach (Atomflot),NaN,NaN,06.08.16 7:00,21.07.16 18:30,15.5
3,Yong Sheng,Hong Kong,Cosco Shipping Ltd,Arc 4,General cargo carrier,Ore in bulk.steel,13 514,19 150,8.3,Shanghai (China),Glasgow (Scotland),Vaigach (Atomflot),NaN,NaN,06.08.16 13:26,28.07.16 7:00,9.3
4,Spasatel Karev,Russia,FBI “Marine Rescue Service of Rosmorrechflot”,Arc 5,Rescue,NaN,0,1 109,NaN,Saint- Petersbourg,Vladivostok,No,NaN,30.07.16 6:30,NaN,20.08.16 9:50,21.1
5,Gretke Oldendorf,Portugal,Oldendorff Carriers GmbH & Co KG,Ice 2,General cargo carrier,Coal,76 175,80 444,14.3,Vancouver (Canada),Raahe (Finland),Yamal (Atomflot),NaN,NaN,28.08.2016 23:30,03.08.16 6:00,25.7


In [79]:
# expand columns 

df_2016[['Kara Gate date','Kara Gate time']] = df_2016['Kara Gate date and time'].str.split(n=1,expand=True)

df_2016[['Yugorskiy Strait date','Yugorskiy Strait time']] = df_2016['Yugorskiy Strait date and time'].str.split(n=1,expand=True)

df_2016[['Cape Zhelaniya date','Cape Zhelaniya time']] = df_2016['Cape Zhelaniya date and time'].str.split(n=1,expand=True)

df_2016[['Cape Dezhnev date','Cape Dezhnev time']] = df_2016['Cape Dezhnev date and time'].str.split(n=1,expand=True)

df_2016.drop(['Kara Gate date and time','Yugorskiy Strait date and time','Cape Zhelaniya date and time','Cape Dezhnev date and time'],axis=1,inplace=True)

In [80]:
df_2016.head()

,vessel name,flag,ship operator,ice class,ship type,cargo type,approx cargo weight (tn),dead weight tonnage,max draught,port of departure,...,icebreaker assistance,time elapsed on nsr,Kara Gate date,Kara Gate time,Yugorskiy Strait date,Yugorskiy Strait time,Cape Zhelaniya date,Cape Zhelaniya time,Cape Dezhnev date,Cape Dezhnev time
1,Kapitan Khlebnikov,Russia,"PJSC ""Far Eastern Shipping Company""",Icebreaker 7,Icebreaker,Passangers,120,4 418,8.5,Anadyr,...,No,11.8,NaN,NaN,NaN,NaN,27.07.16,21:35,16.07.16,2:30
2,Brage Viking,Russia,"LLC ""Viking Supply""",Arc 5,Tug,NaN,0,4 352,7.4,Kholmsk,...,Vaigach (Atomflot),15.5,NaN,NaN,NaN,NaN,06.08.16,7:00,21.07.16,18:30
3,Yong Sheng,Hong Kong,Cosco Shipping Ltd,Arc 4,General cargo carrier,Ore in bulk.steel,13 514,19 150,8.3,Shanghai (China),...,Vaigach (Atomflot),9.3,NaN,NaN,NaN,NaN,06.08.16,13:26,28.07.16,7:00
4,Spasatel Karev,Russia,FBI “Marine Rescue Service of Rosmorrechflot”,Arc 5,Rescue,NaN,0,1 109,NaN,Saint- Petersbourg,...,No,21.1,NaN,NaN,30.07.16,6:30,NaN,NaN,20.08.16,9:50
5,Gretke Oldendorf,Portugal,Oldendorff Carriers GmbH & Co KG,Ice 2,General cargo carrier,Coal,76 175,80 444,14.3,Vancouver (Canada),...,Yamal (Atomflot),25.7,NaN,NaN,NaN,NaN,28.08.2016,23:30,03.08.16,6:00


In [81]:
# 2017

df_2017 = data[2017]

df_2017.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,0,N,Vessel name,DWT,GRT,Flag,Shipowner,Vessel type,IMO,LOA,Ice class,Port of exit,Exit date,NSR entered,Entry date,NSR left,Leaving\ndate,Port of entry,Entry date,NSR passage\ntime (days)
1,1,1,Tian Jian,38146,26770,China,COSCO,Heavy Lift,9722754,190,Ice 1,Lianyungang,31.08.2017 7:00,Cape Dezhnev,10.09.17,Cape Zhelaniya,17.09.17,Esbjerg,24.09.17,"6,6"
2,2,2,Da An,28000,22566,Hong Kong,COSCO,Heavy Lift,9607825,"179,5",Ice 1,Tianjin,22.08.2017 2:00,Cape Dezhnev,01.09.17,Cape Zhelaniya,07.09.17,Cuxhaven,14.09.17,"6,1"
3,3,3,Foresight,12669,9611,Antigua and Barbuda,BBC,General cargo,9388912,"138,8",Arc 4,Taicang,08.08.2017 10:00,Cape Dezhnev,21.08.17,Kara Gate,29.08.17,Bermerhaven,04.09.17,"7,9"
4,4,4,Fortune,12692,9611,Antigua and Barbuda,BBC,General cargo,9402067,139,Arc 4,Tiacang,11.09.2017 1:00,Cape Dezhnev,22.09.17,Cape Zhelaniya,30.09.17,Brake,07.10.17,"7,8"


In [82]:
df_2017.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
23,23,23,Egvekinot,6026,4110,Russia,Oil-Compact,Tanker,7912044,"106,1",Arc 4,Murmansk,13.08.2017 22:40,Kara Gate,15.08.17,Cape Dezhnev,26.08.17,Provideniya,29.08.17,"10,9"
24,24,24,HHL Elbe,12709,9627,Liberia,HHL,General cargo,9433262,"138,5",Arc 4,Aberdeen,14.07.2017 3:00,Kara Gate,19.07.17,Cape Dezhnev,29.07.17,Kholmsk,05.08.17,"10,4"
25,25,25,Siziman,4800,4464,Sierra Leone,Garten Holding,General cargo,9113604,"117,6",Arc 4,Murmansk,04.09.2017 23:00,Kara Gate,07.09.17,Cape Dezhnev,20.09.17,Lanshan,09.10.17,"12,9"
26,26,26,Rolldock Storm,9063,15382,Netherlands,RollDock Shipping,Heavy load,9656503,"151,5",Ice 3,Severodvinsk,18.09.2017 9:00,Kara Gate,20.09.17,Cape Dezhnev,27.09.17,Qingdao,08.10.17,"7,4"
27,27,27,Garmoniya,3642,4295,Russia,Yuzhmorrybflot,Reefer,8516639,103,Arc 4,Ossora,07.08.2017 1:30,Cape Dezhnev,11.08.17,Kara Gate,21.08.17,Arkhangelsk,23.08.17,"10,0"


In [83]:
df_2017.columns = ['index','index2','vessel name','dead weight tonnage','grt',
                   'flag','ship operator','ship type','imo ship id','length overall',
                   'ice class','port of departure','departure date and time',
                   'nsr entrance point','nsr entry date','nsr exit point','nsr exit date',
                   'port of arrival','date of arrival','time elapsed on nsr']

df_2017.head()

,index,index2,vessel name,dead weight tonnage,grt,flag,ship operator,ship type,imo ship id,length overall,ice class,port of departure,departure date and time,nsr entrance point,nsr entry date,nsr exit point,nsr exit date,port of arrival,date of arrival,time elapsed on nsr
0,0,N,Vessel name,DWT,GRT,Flag,Shipowner,Vessel type,IMO,LOA,Ice class,Port of exit,Exit date,NSR entered,Entry date,NSR left,Leaving\ndate,Port of entry,Entry date,NSR passage\ntime (days)
1,1,1,Tian Jian,38146,26770,China,COSCO,Heavy Lift,9722754,190,Ice 1,Lianyungang,31.08.2017 7:00,Cape Dezhnev,10.09.17,Cape Zhelaniya,17.09.17,Esbjerg,24.09.17,"6,6"
2,2,2,Da An,28000,22566,Hong Kong,COSCO,Heavy Lift,9607825,"179,5",Ice 1,Tianjin,22.08.2017 2:00,Cape Dezhnev,01.09.17,Cape Zhelaniya,07.09.17,Cuxhaven,14.09.17,"6,1"
3,3,3,Foresight,12669,9611,Antigua and Barbuda,BBC,General cargo,9388912,"138,8",Arc 4,Taicang,08.08.2017 10:00,Cape Dezhnev,21.08.17,Kara Gate,29.08.17,Bermerhaven,04.09.17,"7,9"
4,4,4,Fortune,12692,9611,Antigua and Barbuda,BBC,General cargo,9402067,139,Arc 4,Tiacang,11.09.2017 1:00,Cape Dezhnev,22.09.17,Cape Zhelaniya,30.09.17,Brake,07.10.17,"7,8"


In [84]:
# drop unnecessary columns

df_2017.drop(['index','index2'],axis=1,inplace=True)

df_2017.drop(0,axis=0,inplace=True)

df_2017.head()

,vessel name,dead weight tonnage,grt,flag,ship operator,ship type,imo ship id,length overall,ice class,port of departure,departure date and time,nsr entrance point,nsr entry date,nsr exit point,nsr exit date,port of arrival,date of arrival,time elapsed on nsr
1,Tian Jian,38146,26770,China,COSCO,Heavy Lift,9722754,190,Ice 1,Lianyungang,31.08.2017 7:00,Cape Dezhnev,10.09.17,Cape Zhelaniya,17.09.17,Esbjerg,24.09.17,"6,6"
2,Da An,28000,22566,Hong Kong,COSCO,Heavy Lift,9607825,"179,5",Ice 1,Tianjin,22.08.2017 2:00,Cape Dezhnev,01.09.17,Cape Zhelaniya,07.09.17,Cuxhaven,14.09.17,"6,1"
3,Foresight,12669,9611,Antigua and Barbuda,BBC,General cargo,9388912,"138,8",Arc 4,Taicang,08.08.2017 10:00,Cape Dezhnev,21.08.17,Kara Gate,29.08.17,Bermerhaven,04.09.17,"7,9"
4,Fortune,12692,9611,Antigua and Barbuda,BBC,General cargo,9402067,139,Arc 4,Tiacang,11.09.2017 1:00,Cape Dezhnev,22.09.17,Cape Zhelaniya,30.09.17,Brake,07.10.17,"7,8"
5,Finesse,12685,9611,Antigua and Barbuda,BBC,General cargo,9388900,138,Arc 4,Taicang,27.09.2017 0:45,Cape Dezhnev,07.10.17,Cape Zhelaniya,14.10.17,Brake,21.10.17,"6,9"


In [85]:
df_2017 = basic_clean(df_2017)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name                0
dead weight tonnage        0
grt                        0
flag                       0
ship operator              0
ship type                  0
imo ship id                0
length overall             0
ice class                  0
port of departure          0
departure date and time    0
nsr entrance point         0
nsr entry date             0
nsr exit point             0
nsr exit date              0
port of arrival            0
date of arrival            0
time elapsed on nsr        0
dtype: int64


In [86]:
# expand columns
df_2017[['date of departure','time of departure']] = df_2017['departure date and time'].str.split(n=1,expand=True)
df_2017.drop('departure date and time',axis=1,inplace=True)

In [87]:
df_2017.tail()

,vessel name,dead weight tonnage,grt,flag,ship operator,ship type,imo ship id,length overall,ice class,port of departure,nsr entrance point,nsr entry date,nsr exit point,nsr exit date,port of arrival,date of arrival,time elapsed on nsr,date of departure,time of departure
23,Egvekinot,6026,4110,Russia,Oil-Compact,Tanker,7912044,106.1,Arc 4,Murmansk,Kara Gate,15.08.17,Cape Dezhnev,26.08.17,Provideniya,29.08.17,10.9,13.08.2017,22:40
24,HHL Elbe,12709,9627,Liberia,HHL,General cargo,9433262,138.5,Arc 4,Aberdeen,Kara Gate,19.07.17,Cape Dezhnev,29.07.17,Kholmsk,05.08.17,10.4,14.07.2017,3:00
25,Siziman,4800,4464,Sierra Leone,Garten Holding,General cargo,9113604,117.6,Arc 4,Murmansk,Kara Gate,07.09.17,Cape Dezhnev,20.09.17,Lanshan,09.10.17,12.9,04.09.2017,23:00
26,Rolldock Storm,9063,15382,Netherlands,RollDock Shipping,Heavy load,9656503,151.5,Ice 3,Severodvinsk,Kara Gate,20.09.17,Cape Dezhnev,27.09.17,Qingdao,08.10.17,7.4,18.09.2017,9:00
27,Garmoniya,3642,4295,Russia,Yuzhmorrybflot,Reefer,8516639,103,Arc 4,Ossora,Cape Dezhnev,11.08.17,Kara Gate,21.08.17,Arkhangelsk,23.08.17,10.0,07.08.2017,1:30


In [88]:
# 2018

df_2018 = data[2018]
df_2018.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0,NaN,Vessel,NaN,NaN,NaN,NaN,NaN,Departure,NaN,NaN,NSR,NaN,NaN,NaN,Arrival,NaN,NaN
1,1,NaN,Vessel name,Iceclass,Flag,Owner,Vessel type,GRT,Country,Port,Date,Point of entry,Entry,Point of exit,Exit,Country,Port,Date
2,2,1.0,Tian Hui,Arc 4,China,COSCO,General\ncargo,26787,Germany,Emden,05.07.18,C.Zhelaniya,24.07.18,C.Dezhnev,03.08.18,Japan,Tomakomai,12.08.18
3,3,2.0,Tian You,Arc 4,China,COSCO,General\ncargo,26787,S Korea,Busan,20.07.18,C.Dezhnev,28.07.18,C.Zhelaniya,11.08.18,Sweden,Harnosand,20.08.18
4,4,3.0,Tian Jian,Ice 1,China,COSCO,Heavy Load,26770,Denmark,Skagen,30.07.18,C.Zhelaniya,05.08.18,C.Dezhnev,19.08.18,China,Qingdao,02.09.18


In [89]:
df_2018.tail()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
24,24,23.0,Tian Hui,Arc 4,China,COSCO,General\ncargo,26787,China,Jiangsu,20.09.18,C.Dezhnev,30.09.18,C.Zhelaniya,06.10.18,Sweden,Oskarshamn,15.10.18
25,25,24.0,Egvekinot,Arc 4,Russia,Oil-Compact,Tanker,4110,Russia,Arkhangelsk,05.09.18,Kara Gate,08.09.18,C.Dezhnev,19.09.18,Russia,Provideniya,20.09.18
26,26,25.0,Tian You,Arc 4,China,COSCO,General\ncargo,26787,Norway,Hyefjorden,28.09.18,C.Zhelaniya,04.10.18,C.Dezhnev,11.10.18,China,Nansha,25.10.18
27,27,26.0,Argo,Ice 2,Russia,Neva-Charter,Tug,467,Russia,P.Kamchatskiy,24.09.18,C.Dezhnev,02.10.18,Yug.Shar,15.10.18,Russia,Arkhangelsk,19.10.18
28,28,27.0,Novorossiysk,ib 6,Russia,Rosmorport,Icebreaker,13867,Russia,Murmansk,22.11.18,Kara Gate,24.11.18,C.Dezhnev,03.12.18,Russia,Vanino,17.12.18


In [90]:
df_2018.columns = ['index','index2','vessel name','ice class','flag','ship operator','ship type',
                   'grt','country of departure','port of departure','date of departure',
                   'nsr entry point','nsr entry date','nsr exit point','nsr exit date',
                   'country of arrival','port of arrival','date of arrival']
df_2018.head()

,index,index2,vessel name,ice class,flag,ship operator,ship type,grt,country of departure,port of departure,date of departure,nsr entry point,nsr entry date,nsr exit point,nsr exit date,country of arrival,port of arrival,date of arrival
0,0,NaN,Vessel,NaN,NaN,NaN,NaN,NaN,Departure,NaN,NaN,NSR,NaN,NaN,NaN,Arrival,NaN,NaN
1,1,NaN,Vessel name,Iceclass,Flag,Owner,Vessel type,GRT,Country,Port,Date,Point of entry,Entry,Point of exit,Exit,Country,Port,Date
2,2,1.0,Tian Hui,Arc 4,China,COSCO,General\ncargo,26787,Germany,Emden,05.07.18,C.Zhelaniya,24.07.18,C.Dezhnev,03.08.18,Japan,Tomakomai,12.08.18
3,3,2.0,Tian You,Arc 4,China,COSCO,General\ncargo,26787,S Korea,Busan,20.07.18,C.Dezhnev,28.07.18,C.Zhelaniya,11.08.18,Sweden,Harnosand,20.08.18
4,4,3.0,Tian Jian,Ice 1,China,COSCO,Heavy Load,26770,Denmark,Skagen,30.07.18,C.Zhelaniya,05.08.18,C.Dezhnev,19.08.18,China,Qingdao,02.09.18


In [91]:
df_2018.drop(['index','index2'],axis=1,inplace=True)
df_2018.drop([0,1],axis=0,inplace=True)
df_2018.reset_index(drop=True,inplace=True)
df_2018.head()

,vessel name,ice class,flag,ship operator,ship type,grt,country of departure,port of departure,date of departure,nsr entry point,nsr entry date,nsr exit point,nsr exit date,country of arrival,port of arrival,date of arrival
0,Tian Hui,Arc 4,China,COSCO,General\ncargo,26787,Germany,Emden,05.07.18,C.Zhelaniya,24.07.18,C.Dezhnev,03.08.18,Japan,Tomakomai,12.08.18
1,Tian You,Arc 4,China,COSCO,General\ncargo,26787,S Korea,Busan,20.07.18,C.Dezhnev,28.07.18,C.Zhelaniya,11.08.18,Sweden,Harnosand,20.08.18
2,Tian Jian,Ice 1,China,COSCO,Heavy Load,26770,Denmark,Skagen,30.07.18,C.Zhelaniya,05.08.18,C.Dezhnev,19.08.18,China,Qingdao,02.09.18
3,Atlantic Winter,Arc 4,Portugal,Atlantic Winter,General\ncargo,15549,China,Taicang,30.07.18,C.Dezhnev,08.08.18,C.Zhelaniya,16.08.18,Sweden,Harnosand,31.08.18
4,Kairos,No,Russia,Morskoy\nStandard,Tanker,27526,Russia,P.Kamchatskiy,08.08.18,C.Dezhnev,11.08.18,Kara Gate,23.08.18,Russia,Murmansk,29.08.18


In [92]:
df_2018 = basic_clean(df_2018)
df_2018.head()

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name             0
ice class               0
flag                    0
ship operator           0
ship type               0
grt                     0
country of departure    0
port of departure       0
date of departure       0
nsr entry point         0
nsr entry date          0
nsr exit point          0
nsr exit date           0
country of arrival      0
port of arrival         0
date of arrival         0
dtype: int64


,vessel name,ice class,flag,ship operator,ship type,grt,country of departure,port of departure,date of departure,nsr entry point,nsr entry date,nsr exit point,nsr exit date,country of arrival,port of arrival,date of arrival
0,Tian Hui,Arc 4,China,COSCO,General cargo,26787,Germany,Emden,05.07.18,C.Zhelaniya,24.07.18,C.Dezhnev,03.08.18,Japan,Tomakomai,12.08.18
1,Tian You,Arc 4,China,COSCO,General cargo,26787,S Korea,Busan,20.07.18,C.Dezhnev,28.07.18,C.Zhelaniya,11.08.18,Sweden,Harnosand,20.08.18
2,Tian Jian,Ice 1,China,COSCO,Heavy Load,26770,Denmark,Skagen,30.07.18,C.Zhelaniya,05.08.18,C.Dezhnev,19.08.18,China,Qingdao,02.09.18
3,Atlantic Winter,Arc 4,Portugal,Atlantic Winter,General cargo,15549,China,Taicang,30.07.18,C.Dezhnev,08.08.18,C.Zhelaniya,16.08.18,Sweden,Harnosand,31.08.18
4,Kairos,No,Russia,Morskoy Standard,Tanker,27526,Russia,P.Kamchatskiy,08.08.18,C.Dezhnev,11.08.18,Kara Gate,23.08.18,Russia,Murmansk,29.08.18


In [93]:
# 2019

df_2019 = data[2019]
df_2019.head()


,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
0,Tian En,9774587,Arc 4,China,COSCO,General cargo,37125,26787,190.0,China,Changshu,2019-07-19,Sweden,Gavle,2019-08-17
1,Svyatoy Petr,9035838,Arc 4,Russia,Valkur,Tanker,10960,6441,127.2,Russia,Petropavlovsk-Kamchatskiy,2019-07-23,Russia,Arkhangelsk,2019-08-10
2,Professor Multanovskiy,8010362,Arc 5,Russia,FERHRI,Research,620,1753,72.0,Russia,Vladivostok,2019-07-26,Russia,Murmansk,2019-09-07
3,Da Tai,9608386,Ice 1,Hong Kong,COSCO,General cargo,29881,22566,179.5,China,Taicang,2019-07-31,Denmark,Skagen,2019-08-24
4,Simfoniya,8516627,Arc 4,Russia,Transrefflot,Reefer,3750,4298,103.0,Russia,Ossora,2019-08-06,Russia,Arkhangelsk,2019-08-19


In [94]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Vessel name       36 non-null     object 
 1   IMO               36 non-null     int64  
 2   Ice class         36 non-null     object 
 3   Flag              36 non-null     object 
 4   Owner / Operator  36 non-null     object 
 5   Vessel type       36 non-null     object 
 6   DWT               36 non-null     int64  
 7   GRT               36 non-null     int64  
 8   LOA               36 non-null     float64
 9   Dep Country       36 non-null     object 
 10  Dep Port          36 non-null     object 
 11  Dep Date          36 non-null     object 
 12  Arr Country       36 non-null     object 
 13  Arr Port          36 non-null     object 
 14  Arr Date          36 non-null     object 
dtypes: float64(1), int64(3), object(11)
memory usage: 4.3+ KB


In [95]:
df_2019.columns = ['vessel name','imo ship id','ice class','flag','ship operator','ship type','dead weight tonnage','grt','length overall','country of departure','port of departure','date of departure','country of arrival','port of arrival','date of arrival']

In [96]:
df_2019.head()

,vessel name,imo ship id,ice class,flag,ship operator,ship type,dead weight tonnage,grt,length overall,country of departure,port of departure,date of departure,country of arrival,port of arrival,date of arrival
0,Tian En,9774587,Arc 4,China,COSCO,General cargo,37125,26787,190.0,China,Changshu,2019-07-19,Sweden,Gavle,2019-08-17
1,Svyatoy Petr,9035838,Arc 4,Russia,Valkur,Tanker,10960,6441,127.2,Russia,Petropavlovsk-Kamchatskiy,2019-07-23,Russia,Arkhangelsk,2019-08-10
2,Professor Multanovskiy,8010362,Arc 5,Russia,FERHRI,Research,620,1753,72.0,Russia,Vladivostok,2019-07-26,Russia,Murmansk,2019-09-07
3,Da Tai,9608386,Ice 1,Hong Kong,COSCO,General cargo,29881,22566,179.5,China,Taicang,2019-07-31,Denmark,Skagen,2019-08-24
4,Simfoniya,8516627,Arc 4,Russia,Transrefflot,Reefer,3750,4298,103.0,Russia,Ossora,2019-08-06,Russia,Arkhangelsk,2019-08-19


In [97]:
df_2019 = basic_clean(df_2019)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name             0
imo ship id             0
ice class               0
flag                    0
ship operator           0
ship type               0
dead weight tonnage     0
grt                     0
length overall          0
country of departure    0
port of departure       0
date of departure       0
country of arrival      0
port of arrival         0
date of arrival         0
dtype: int64


In [98]:
# 2020

df_2020 = data[2020]

df_2020.head()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
0,Barents,9278600,Arc 4,Russia,Eco Shipping,General cargo,4303,3198,98.9,Russia,Severodvinsk,2020-07-17,Russia,Bolshoy Kamen,2020-08-07
1,"S,Kuznetsov",9210359,Arc 4,Russia,Northern Shipping Company,General cargo,7625,6204,107.8,Russia,Belomorsk,2020-07-20,Russia,Petropavlovsk-Kamchatskiy,2020-08-06
2,Nordic Svalbard,9437323,Arc 4,Liberia,Nordic Svalbard,General cargo,16543,15549,133.0,Norway,Jossingfjord,2020-07-21,Russia,Vladivostok,2020-08-12
3,Tian En,9774587,Arc 4,China,COSCO,General cargo,37125,26787,190.0,China,Lianyungang,2020-07-26,Germany,Emden,2020-08-19
4,Samson,9114751,Arc 4,Russia,Morskoy Standard-Bunker,Tanker,5870,3726,99.8,Russia,Arkhangelsk,2020-07-27,Russia,Bering Sea,2020-08-07


In [99]:
df_2020.tail()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
59,Golden Strength,9413420,Ice 2,Marshall Islands,Suek AG,Bulk,75744,41655,225.00,Russia,Murmansk,2020-10-21,China,Tianjin,2020-11-17
60,UHL Partner,9418999,Arc 4,Portugal,United Heavy Lift,General cargo,16523,15549,166.14,Russia,Ust-Luga,2020-10-25,Russia,Vostochnyy,2020-11-23
61,Golden Pearl,9470375,Ice 2,Marshall Islands,Suek AG,Bulk,74300,41718,225.00,Russia,Murmansk,2020-10-27,China,Caofeidian,2020-11-20
62,Boris Vilkitskiy (Tanker),9000247,Arc 4,Russia,Khatanga Sea Port,Tanker,6760,5025,108.40,Russia,Beringovskiy,2020-10-30,Russia,Murmansk,2020-11-14
63,SCF Sakhalin,9307724,Icebreaker 6,Russia,SCF,Supply,4298,6882,100.00,Russia,Kholmsk,2020-10-31,Russia,Murmansk,2020-11-20


In [100]:
df_2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Vessel name       64 non-null     object 
 1   IMO               64 non-null     int64  
 2   Ice class         64 non-null     object 
 3   Flag              64 non-null     object 
 4   Owner / Operator  64 non-null     object 
 5   Vessel type       64 non-null     object 
 6   DWT               64 non-null     int64  
 7   GRT               64 non-null     int64  
 8   LOA               64 non-null     float64
 9   Dep Country       64 non-null     object 
 10  Dep Port          64 non-null     object 
 11  Dep Date          64 non-null     object 
 12  Arr Country       64 non-null     object 
 13  Arr Port          64 non-null     object 
 14  Arr Date          64 non-null     object 
dtypes: float64(1), int64(3), object(11)
memory usage: 7.6+ KB


In [101]:
df_2020.columns = ['vessel name','imo ship id','ice class','flag','ship operator','ship type','dead weight tonnage','grt','length overall','country of departure','port of departure','date of departure','country of arrival','port of arrival','date of arrival']
df_2020.head()

,vessel name,imo ship id,ice class,flag,ship operator,ship type,dead weight tonnage,grt,length overall,country of departure,port of departure,date of departure,country of arrival,port of arrival,date of arrival
0,Barents,9278600,Arc 4,Russia,Eco Shipping,General cargo,4303,3198,98.9,Russia,Severodvinsk,2020-07-17,Russia,Bolshoy Kamen,2020-08-07
1,"S,Kuznetsov",9210359,Arc 4,Russia,Northern Shipping Company,General cargo,7625,6204,107.8,Russia,Belomorsk,2020-07-20,Russia,Petropavlovsk-Kamchatskiy,2020-08-06
2,Nordic Svalbard,9437323,Arc 4,Liberia,Nordic Svalbard,General cargo,16543,15549,133.0,Norway,Jossingfjord,2020-07-21,Russia,Vladivostok,2020-08-12
3,Tian En,9774587,Arc 4,China,COSCO,General cargo,37125,26787,190.0,China,Lianyungang,2020-07-26,Germany,Emden,2020-08-19
4,Samson,9114751,Arc 4,Russia,Morskoy Standard-Bunker,Tanker,5870,3726,99.8,Russia,Arkhangelsk,2020-07-27,Russia,Bering Sea,2020-08-07


In [102]:
df_2020 = basic_clean(df_2020)


New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name             0
imo ship id             0
ice class               0
flag                    0
ship operator           0
ship type               0
dead weight tonnage     0
grt                     0
length overall          0
country of departure    0
port of departure       0
date of departure       0
country of arrival      0
port of arrival         0
date of arrival         0
dtype: int64


In [103]:
# 2021

df_2021 = data[2021]

df_2021.head()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
0,Tian En,9774587,Arc 4,China,COSCO,General cargo,37125,26787,190.0,Finland,Helsinki,2021-06-23,China,Qingdao,2021-08-17
1,Admiral Shmidt,9838838,Arc 5,Bahamas,GTLK Asia M3,Bulk,104553,66291,250.0,Russia,Nakhodka,2021-07-07,Canada,Nunavut,2021-08-11
2,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206.3,Netherlands,Rotterdam,2021-07-09,China,Yantai,2021-08-14
3,Golden Furious,9860128,Ice 2,Marshall Islands,Furious Shipowning,Bulk,81600,44094,229.0,Russia,Vanino,2021-07-10,Russia,Murmansk,2021-08-06
4,Svyatoy Pavel,9035826,Arc 4,Russia,Valkur,Tanker,10926,6441,127.2,Russia,Beringovskiy,2021-07-15,Russia,Arkhangelsk,2021-08-03


In [104]:
df_2021.tail()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
82,UHL Fusion,9785380,Arc 4,Portugal,Elmar Shipping,General cargo,17147,16732,150.0,Denmark,Esbjerg,2021-10-25,Russia,Bolshoy Kamen,2021-11-29
83,Golden Pearl,9470375,Ice 2,Marshall Islands,Suek AG,Bulk,74300,41718,225.0,Russia,Murmansk,2021-10-26,China,Rizhao,2021-12-02
84,Kumpula,9590802,Arc 4,Finland,Suek AG,Bulk,56348,33958,197.0,China,Rizhao,2021-10-26,Norway,Narvik,2021-12-11
85,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260.0,Russia,"St, Petersburg",2021-11-02,Russia,Nakhodka,2021-11-24
86,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206.3,China,Nantong,2021-11-06,Russia,Murmansk,2022-01-10


In [105]:
df_2021.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87 entries, 0 to 86
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Vessel name       87 non-null     object 
 1   IMO               87 non-null     int64  
 2   Ice class         87 non-null     object 
 3   Flag              87 non-null     object 
 4   Owner / Operator  87 non-null     object 
 5   Vessel type       87 non-null     object 
 6   DWT               87 non-null     int64  
 7   GRT               87 non-null     int64  
 8   LOA               87 non-null     float64
 9   Dep Country       87 non-null     object 
 10  Dep Port          87 non-null     object 
 11  Dep Date          87 non-null     object 
 12  Arr Country       87 non-null     object 
 13  Arr Port          87 non-null     object 
 14  Arr Date          87 non-null     object 
dtypes: float64(1), int64(3), object(11)
memory usage: 10.3+ KB


In [106]:
df_2021.columns = ['vessel name','imo ship id','ice class','flag','ship operator','ship type','dead weight tonnage','grt','length overall','country of departure','port of departure','date of departure','country of arrival','port of arrival','date of arrival']
df_2021.head()

,vessel name,imo ship id,ice class,flag,ship operator,ship type,dead weight tonnage,grt,length overall,country of departure,port of departure,date of departure,country of arrival,port of arrival,date of arrival
0,Tian En,9774587,Arc 4,China,COSCO,General cargo,37125,26787,190.0,Finland,Helsinki,2021-06-23,China,Qingdao,2021-08-17
1,Admiral Shmidt,9838838,Arc 5,Bahamas,GTLK Asia M3,Bulk,104553,66291,250.0,Russia,Nakhodka,2021-07-07,Canada,Nunavut,2021-08-11
2,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206.3,Netherlands,Rotterdam,2021-07-09,China,Yantai,2021-08-14
3,Golden Furious,9860128,Ice 2,Marshall Islands,Furious Shipowning,Bulk,81600,44094,229.0,Russia,Vanino,2021-07-10,Russia,Murmansk,2021-08-06
4,Svyatoy Pavel,9035826,Arc 4,Russia,Valkur,Tanker,10926,6441,127.2,Russia,Beringovskiy,2021-07-15,Russia,Arkhangelsk,2021-08-03


In [107]:
df_2021 = basic_clean(df_2021)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name             0
imo ship id             0
ice class               0
flag                    0
ship operator           0
ship type               0
dead weight tonnage     0
grt                     0
length overall          0
country of departure    0
port of departure       0
date of departure       0
country of arrival      0
port of arrival         0
date of arrival         0
dtype: int64


In [108]:
# 2022

df_2022 = data[2022]

df_2022.head()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
0,Audax,9763837,Arc 7,Curacao,"Fortune MC Hercules Shipping Ltd,",General cargo,28899,34387,206,China,Tianjin,2021-12-29,Russia,Murmansk,2022-02-17
1,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,Vladivostok,2022-01-17,Russia,Murmansk,2022-02-23
2,Ruslana,9352339,Arc 4,Russia,Transflot,General cargo,7750,5057,120,South Korea,Masan,2022-06-25,Russia,Ust-Luga,2022-08-05
3,Svyatoy Petr,9035838,Arc 4,Russia,Valkur,Tanker,10960,6441,127,Russia,Beringovskiy,2022-07-06,Russia,Arkhangelsk,2022-07-30
4,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,Murmansk,2022-07-08,Russia,Petropavlovsk-Kamchatskiy,2022-07-21


In [109]:
df_2022.tail()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date
38,PANDAR,9000235,Ice 2,Russia,Morskoy Standart Co Ltd,Tanker,6810,5025,116,Russia,Petropavlovsk-Kamchatskiy,2022-10-09,Russia,Arkhangelsk,2022-11-02
39,ELLADORA,8822375,Ice 1,Russia,DEFA Fishing LLC,Fishing,1919,1523,60,Norway,Kirkenes,2022-10-21,Russia,Vladivostok,2022-11-22
40,VASILY DINKOV,9372547,Arc 6,Russia,SOVCOMFLOT,Tanker,72722,49597,257,Russia,Murmansk,2022-10-27,China,Rizhao,2022-11-23
41,Monchegorsk,9404015,Arc 7,Russia,Norilsk Nickel,Container,18486,16994,169,Russia,Nakhodka,2022-11-01,Russia,Murmansk,2022-11-24
42,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,Nakhodka,2022-11-10,Russia,"St, Petersburg",2022-12-15


In [110]:
df_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Vessel name       43 non-null     object
 1   IMO               43 non-null     int64 
 2   Ice class         43 non-null     object
 3   Flag              43 non-null     object
 4   Owner / Operator  43 non-null     object
 5   Vessel type       43 non-null     object
 6   DWT               43 non-null     int64 
 7   GRT               43 non-null     int64 
 8   LOA               43 non-null     int64 
 9   Dep Country       43 non-null     object
 10  Dep Port          43 non-null     object
 11  Dep Date          43 non-null     object
 12  Arr Country       43 non-null     object
 13  Arr Port          43 non-null     object
 14  Arr Date          43 non-null     object
dtypes: int64(4), object(11)
memory usage: 5.2+ KB


In [111]:
df_2022.columns = ['vessel name','imo ship id','ice class','flag','ship operator','ship type','dead weight tonnage','grt','length overall','country of departure','port of departure','date of departure','country of arrival','port of arrival','date of arrival']
df_2022.head()

,vessel name,imo ship id,ice class,flag,ship operator,ship type,dead weight tonnage,grt,length overall,country of departure,port of departure,date of departure,country of arrival,port of arrival,date of arrival
0,Audax,9763837,Arc 7,Curacao,"Fortune MC Hercules Shipping Ltd,",General cargo,28899,34387,206,China,Tianjin,2021-12-29,Russia,Murmansk,2022-02-17
1,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,Vladivostok,2022-01-17,Russia,Murmansk,2022-02-23
2,Ruslana,9352339,Arc 4,Russia,Transflot,General cargo,7750,5057,120,South Korea,Masan,2022-06-25,Russia,Ust-Luga,2022-08-05
3,Svyatoy Petr,9035838,Arc 4,Russia,Valkur,Tanker,10960,6441,127,Russia,Beringovskiy,2022-07-06,Russia,Arkhangelsk,2022-07-30
4,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,Murmansk,2022-07-08,Russia,Petropavlovsk-Kamchatskiy,2022-07-21


In [112]:
df_2022 = basic_clean(df_2022)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name             0
imo ship id             0
ice class               0
flag                    0
ship operator           0
ship type               0
dead weight tonnage     0
grt                     0
length overall          0
country of departure    0
port of departure       0
date of departure       0
country of arrival      0
port of arrival         0
date of arrival         0
dtype: int64


In [113]:
# 2023

df_2023 = data[2023]

df_2023.head()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date,Cargo Approx tn,Cargo type
0,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206,South Korea,Busan,2023-06-03,Russia,Murmansk,2023-07-03,11748.80,general cargo
1,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,"St, Petersburg",2023-06-24,Russia,Vostochnyy,2023-07-21,4764.74,containers
2,Chukotka+,7927960,Arc 4,Russia,Chukotka Trading Company,Tanker,11548,8806,141,Russia,Egvekinot,2023-07-04,Russia,Arkhangelsk,2023-07-26,0.00,ballast
3,SCF Baltica,9305568,Arc 5,Liberia,Ashbourne Navigation,Tanker,117153,65293,250,Russia,Vostochnyy,2023-07-07,Russia,Murmansk,2023-07-29,0.00,ballast
4,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206,Russia,Murmansk,2023-07-07,China,Qidong,2023-08-07,0.00,ballast


In [114]:
df_2023.tail()

,Vessel name,IMO,Ice class,Flag,Owner / Operator,Vessel type,DWT,GRT,LOA,Dep Country,Dep Port,Dep Date,Arr Country,Arr Port,Arr Date,Cargo Approx tn,Cargo type
74,Gingo,9182710,No Ice Class,Liberia,Light Maritime,Bulk,169159,86192,289,China,Dalian,2023-09-22,Russia,Murmansk,2023-11-09,0.0,ballast
75,Almaz,9150224,Ice 1,Russia,MAGE,Tug,57,182,29,Russia,Murmansk,2023-10-14,Russia,Livadiya,2023-11-15,0.0,ballast
76,NS Arctic,9413547,Ice 3,Liberia,Ns Arctic Shipping,Tanker,111107,62433,250,Russia,Primorsk,2023-10-06,China,Tianjin,2023-11-17,99967.8,crude oil
77,Newnew Polar Bear,9313204,Arc 5,Hong Kong,HAINANXIN XIN YANG,Container,15952,16324,169,Russia,Arkhangelsk,2023-10-24,China,Tianjin,2023-12-05,13171.3,containers
78,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,Petropavlovsk-Kamchatskiy,2023-11-15,Russia,"St, Petersburg",2023-12-05,2459.6,containers


In [115]:
df_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Vessel name       79 non-null     object 
 1   IMO               79 non-null     int64  
 2   Ice class         79 non-null     object 
 3   Flag              79 non-null     object 
 4   Owner / Operator  79 non-null     object 
 5   Vessel type       79 non-null     object 
 6   DWT               79 non-null     int64  
 7   GRT               79 non-null     int64  
 8   LOA               79 non-null     int64  
 9   Dep Country       79 non-null     object 
 10  Dep Port          79 non-null     object 
 11  Dep Date          79 non-null     object 
 12  Arr Country       79 non-null     object 
 13  Arr Port          79 non-null     object 
 14  Arr Date          79 non-null     object 
 15  Cargo Approx tn   79 non-null     float64
 16  Cargo type        79 non-null     object 
dtyp

In [116]:
df_2023.columns = ['vessel name','imo ship id','ice class','flag','ship operator','ship type','dead weight tonnage','grt','length overall','country of departure','port of departure','date of departure','country of arrival','port of arrival','date of arrival','approx cargo weight (tn)','cargo type']
df_2023.head()

,vessel name,imo ship id,ice class,flag,ship operator,ship type,dead weight tonnage,grt,length overall,country of departure,port of departure,date of departure,country of arrival,port of arrival,date of arrival,approx cargo weight (tn),cargo type
0,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206,South Korea,Busan,2023-06-03,Russia,Murmansk,2023-07-03,11748.80,general cargo
1,Sevmorput,8729810,Arc 5,Russia,Atomflot,Container,33240,38226,260,Russia,"St, Petersburg",2023-06-24,Russia,Vostochnyy,2023-07-21,4764.74,containers
2,Chukotka+,7927960,Arc 4,Russia,Chukotka Trading Company,Tanker,11548,8806,141,Russia,Egvekinot,2023-07-04,Russia,Arkhangelsk,2023-07-26,0.00,ballast
3,SCF Baltica,9305568,Arc 5,Liberia,Ashbourne Navigation,Tanker,117153,65293,250,Russia,Vostochnyy,2023-07-07,Russia,Murmansk,2023-07-29,0.00,ballast
4,Pugnax,9763849,Arc 7,Liberia,Fortune MC Titan Shipping,Heavy Load,28816,34146,206,Russia,Murmansk,2023-07-07,China,Qidong,2023-08-07,0.00,ballast


In [117]:
df_2023 = basic_clean(df_2023)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name                 0
imo ship id                 0
ice class                   0
flag                        0
ship operator               0
ship type                   0
dead weight tonnage         0
grt                         0
length overall              0
country of departure        0
port of departure           0
date of departure           0
country of arrival          0
port of arrival             0
date of arrival             0
approx cargo weight (tn)    0
cargo type                  0
dtype: int64


In [118]:
# 2024

df_2024 = data[2024]

df_2024.head()

,N,Ship_name,IMO,LOA,DWT,Ice Class,Departure Date,Departure Port,Arrival Date,Arrival port,Arrival Country,Cargo type,Cargo Approx,NSR Entered,NSR Left,Days at NSR
0,1,Audax,9763837,206,28899,Arc 7,2024-01-06 00:00:00,Penglai,2024-02-27 00:00:00,Murmansk,Russia,General cargo,11800,21.01.2024,24.02.2024,34
1,1,Pugnax,9763849,206,28816,Arc 7,2024-01-06 00:00:00,Penglai,2024-02-27 00:00:00,Murmansk,Russia,General cargo,11100,21.01.2024,24.02.2024,34
2,1,Shturman_Skuratov,9759915,249,41041,Arc 7,2024-06-16 00:00:00,Murmansk,2024-07-10 00:00:00,Vostochnyy,Russia,Ballast,0,18.06.2024,29.06.2024,11
3,1,Korolev_Prospect,9826902,250,113232,Ice 3,2024-07-04 00:00:00,Hainan,2024-08-12 00:00:00,Murmansk,Russia,Ballast,0,25.07.2024,09.08.2024,15
4,1,Xue_Long_2,9829241,122,4530,Arc 7,2024-07-05 00:00:00,Qingdao,2024-08-28 00:00:00,Murmansk,Russia,Ballast,0,17.07.2024,23.08.2024,37


In [119]:
df_2024.columns = ['del','vessel name','imo ship id','length overall','dead weight tonnage',
                   'ice class','date of departure','port of departure','date of arrival',
                   'port of arrival','country of arrival','cargo type','approx cargo weight (tn)',
                   'nsr entry date','nsr exit date','del1'
                  ]

df_2024.drop(['del','del1'],axis=1,inplace=True)

In [120]:
df_2024.head()

,vessel name,imo ship id,length overall,dead weight tonnage,ice class,date of departure,port of departure,date of arrival,port of arrival,country of arrival,cargo type,approx cargo weight (tn),nsr entry date,nsr exit date
0,Audax,9763837,206,28899,Arc 7,2024-01-06 00:00:00,Penglai,2024-02-27 00:00:00,Murmansk,Russia,General cargo,11800,21.01.2024,24.02.2024
1,Pugnax,9763849,206,28816,Arc 7,2024-01-06 00:00:00,Penglai,2024-02-27 00:00:00,Murmansk,Russia,General cargo,11100,21.01.2024,24.02.2024
2,Shturman_Skuratov,9759915,249,41041,Arc 7,2024-06-16 00:00:00,Murmansk,2024-07-10 00:00:00,Vostochnyy,Russia,Ballast,0,18.06.2024,29.06.2024
3,Korolev_Prospect,9826902,250,113232,Ice 3,2024-07-04 00:00:00,Hainan,2024-08-12 00:00:00,Murmansk,Russia,Ballast,0,25.07.2024,09.08.2024
4,Xue_Long_2,9829241,122,4530,Arc 7,2024-07-05 00:00:00,Qingdao,2024-08-28 00:00:00,Murmansk,Russia,Ballast,0,17.07.2024,23.08.2024


In [121]:
df_2024.tail()

,vessel name,imo ship id,length overall,dead weight tonnage,ice class,date of departure,port of departure,date of arrival,port of arrival,country of arrival,cargo type,approx cargo weight (tn),nsr entry date,nsr exit date
92,Mikhail_Ulyanov,9333670,236,70000,Arc 6,2024-10-12 00:00:00,Nakhodka,2024-10-27 00:00:00,Prirazlomnaya,Russia,Ballast,0,19.10.2024,27.10.2024
93,Astrum_Apus,9467146,166,19306,Arc 4,2024-10-14 00:00:00,Nakhodka,2024-11-11 00:00:00,Ust-Luga,Russia,General cargo,4100,21.10.2024,30.10.2024
94,Breeze,9305568,250,117153,Arc 5,2024-10-16 00:00:00,Murmansk,2024-11-12 00:00:00,Ningbo,China,Crude Oil,110000,18.10.2024,30.10.2024
95,Viktor_Tsoi,9609744,225,74559,Ice 3,2024-10-18 00:00:00,Murmansk,2024-11-25 00:00:00,Rizhao,China,Coal,71000,22.10.2024,02.11.2024
96,Polar_King,8711320,133,7346,Arc 5,2024-10-21 00:00:00,Arkhangelsk,2024-11-17 00:00:00,P.Kamchatskiy,Russia,General cargo,6200,23.10.2024,13.11.2024


In [122]:
df_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   vessel name               97 non-null     object
 1   imo ship id               97 non-null     int64 
 2   length overall            97 non-null     int64 
 3   dead weight tonnage       97 non-null     int64 
 4   ice class                 97 non-null     object
 5   date of departure         97 non-null     object
 6   port of departure         97 non-null     object
 7   date of arrival           97 non-null     object
 8   port of arrival           97 non-null     object
 9   country of arrival        97 non-null     object
 10  cargo type                97 non-null     object
 11  approx cargo weight (tn)  97 non-null     int64 
 12  nsr entry date            97 non-null     object
 13  nsr exit date             97 non-null     object
dtypes: int64(4), object(10)
memo

In [123]:
df_2024 = basic_clean(df_2024)

New line characters have been removed. Detected 0 cell(s) with non-ASCII characters.
Null Count:
 vessel name                 0
imo ship id                 0
length overall              0
dead weight tonnage         0
ice class                   0
date of departure           0
port of departure           0
date of arrival             0
port of arrival             0
country of arrival          0
cargo type                  0
approx cargo weight (tn)    0
nsr entry date              0
nsr exit date               0
dtype: int64


In [124]:
dfs = [df_2011,df_2012,temp_df_2013,df_2014,df_2015,df_2016,df_2017,df_2018,df_2019,df_2020,df_2021,df_2022,df_2023,df_2024]

In [125]:
df = pd.concat(dfs,ignore_index=True)

In [126]:
df.head()

,vessel name,ship type,flag,cargo type,port of departure,port of arrival,ice class,ship operator,time elapsed on nsr,avg speed,...,Yugorskiy Strait time,Cape Dezhnev time,imo ship id,length overall,nsr entrance point,nsr exit point,date of arrival,country of departure,nsr entry point,country of arrival
0,Perseverance,Tanker vessel,Singapore,Gas condensate,Vitino. Russia,Ningbo. China,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,STI Heritage,Tanker vessel,Marshall Islands,Gas condensate,Vitino. Russia,Map Ta Phut. Thailand,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Marilee,Tanker vessel,Norway,Gas condensate,Vitino. Russia,Huizhou. China,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Vladimir Tikhonov,Tanker vessel,Liberia,Gas condensate,Honningsvåg. Norway,Map Ta Phut. Thailand,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Stena Poseidon,Tanker vessel,Finland,Gas condensate,Vitino. Russia,Incheon. South Korea,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [127]:
df.tail()

,vessel name,ship type,flag,cargo type,port of departure,port of arrival,ice class,ship operator,time elapsed on nsr,avg speed,...,Yugorskiy Strait time,Cape Dezhnev time,imo ship id,length overall,nsr entrance point,nsr exit point,date of arrival,country of departure,nsr entry point,country of arrival
701,Mikhail_Ulyanov,NaN,NaN,Ballast,Nakhodka,Prirazlomnaya,Arc 6,NaN,NaN,NaN,...,NaN,NaN,9333670,236,NaN,NaN,2024-10-27 00:00:00,NaN,NaN,Russia
702,Astrum_Apus,NaN,NaN,General cargo,Nakhodka,Ust-Luga,Arc 4,NaN,NaN,NaN,...,NaN,NaN,9467146,166,NaN,NaN,2024-11-11 00:00:00,NaN,NaN,Russia
703,Breeze,NaN,NaN,Crude Oil,Murmansk,Ningbo,Arc 5,NaN,NaN,NaN,...,NaN,NaN,9305568,250,NaN,NaN,2024-11-12 00:00:00,NaN,NaN,China
704,Viktor_Tsoi,NaN,NaN,Coal,Murmansk,Rizhao,Ice 3,NaN,NaN,NaN,...,NaN,NaN,9609744,225,NaN,NaN,2024-11-25 00:00:00,NaN,NaN,China
705,Polar_King,NaN,NaN,General cargo,Arkhangelsk,P.Kamchatskiy,Arc 5,NaN,NaN,NaN,...,NaN,NaN,8711320,133,NaN,NaN,2024-11-17 00:00:00,NaN,NaN,Russia


In [128]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 706 entries, 0 to 705
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   vessel name               705 non-null    object 
 1   ship type                 440 non-null    object 
 2   flag                      606 non-null    object 
 3   cargo type                409 non-null    object 
 4   port of departure         651 non-null    object 
 5   port of arrival           652 non-null    object 
 6   ice class                 611 non-null    object 
 7   ship operator             513 non-null    object 
 8   time elapsed on nsr       225 non-null    object 
 9   avg speed                 41 non-null     float64
 10  date of departure         561 non-null    object 
 11  time of departure         110 non-null    object 
 12  nsr entry date            318 non-null    object 
 13  nsr entry time            111 non-null    object 
 14  nsr exit d

In [129]:
df['date of departure'].value_counts()

date of departure
2023-09-22             4
2024-09-06 00:00:00    4
2021-09-11             4
2022-09-08             4
2023-07-12             3
2024-07-17 00:00:00    3
2024-08-25 00:00:00    3
2023-08-08             3
2022-09-02             3
2024-09-21 00:00:00    3
2024-09-01 00:00:00    3
2012-07-10 00:00:00    3
2023-07-07             3
2021-08-06             3
2020-09-24             3
2021-07-21             3
2020-09-11             3
2024-09-09 00:00:00    3
2024-09-11 00:00:00    3
2020-09-12             3
2021-09-18             3
2021-10-10             3
03.10.2017             3
2012-10-17 00:00:00    2
13.08.13               2
2023-10-01             2
2024-10-12 00:00:00    2
2024-08-22 00:00:00    2
2023-09-05             2
2012-11-12 00:00:00    2
25.06.13               2
2024-08-28 00:00:00    2
2024-08-30 00:00:00    2
2024-08-16 00:00:00    2
2024-08-21 00:00:00    2
2012-09-30 00:00:00    2
2024-07-30 00:00:00    2
2024-08-14 00:00:00    2
2024-01-06 00:00:00    2
2024-07

In [130]:
df = df.apply(lambda x: x.str.casefold() if x.dtype == 'object' else x)

In [131]:
df['cargo type'].value_counts()

cargo type
ballast                                               131
general cargo                                          34
crude oil                                              32
containers                                             28
 diesel fuel                                           17
oil products tanker                                    12
 general cargo                                         12
gas condensate                                          9
fish                                                    8
crude oil tanker                                        8
diesel fuel                                             8
frozen fish                                             7
coal                                                    7
iron ore concentrate                                    6
diesel oil                                              5
oil/chemical tanker                                     5
gascondensat e novatek                                  5
 na